# BatSpot Fine-Tuning on Kaggle — Detector + Classifier Pipeline

Fine-tune BatSpot pre-trained models on custom bat vocalisation data.

**Pipeline:**
1. **Setup** — Config, dependencies, model/dataset definitions
2. **Data Prep** — Load data, create detector (binary) and classifier (8-class) datasets; recording-leakage check; optional recording-held-out split (per species)
3. **Pretrained Models** — Load detector.pk (192kHz) + classifier.pk (250kHz) from Kaggle upload
4. **Detector Training** — Fine-tune binary detector(s), evaluate
5. **Classifier Training** — Fine-tune 8-class classifier, evaluate
6. **Combined Pipeline** — Classifier alone vs detector hard-gate vs soft combination
7. **Export** — Save the .pk files
8. **Existing Model Evaluation** — Official models scored zero-shot on the same test clips
9. **Inference** — Best detector + classifier over a folder of new recordings → selection-table text file (Cells 21–24)

**Input windowing (important).** Clips are ~375 ms (median) but the CNN reads 20 ms. Training uses a
fresh random 20 ms crop from the loudest part of each clip every epoch; validation/test average the
top-5 loudest non-overlapping windows. The earlier version used only the *first* 20 ms of every clip,
which capped accuracy (classifier ~0.75, detector ~0.85); see `AGENTS.md` ("Windowing fix") for the
measurements. Set `WINDOW_MODE = 'first'` in Cell 2 to reproduce the old behaviour.

In [ ]:
# Cell 2: Configuration
import os

# --- Paths (Kaggle defaults, override as needed) ---
DATA_DIR = '/kaggle/input/datasets/budhil/100eachbatds/Data/Data/final_dataset/data'  # Pre-extracted clips (class subfolders)
RAW_AUDIO_DIR = '/kaggle/input/datasets/budhil/100eachbatds/Data/Data/audio'                                         # Raw .WAV recordings (for clip extraction)
SELECTIONS_DIR = '/kaggle/input/datasets/budhil/100eachbatds/Data/Data/selections'                                        # Raven selection tables (for clip extraction)

# Clip extraction re-reads ALL raw audio and writes into DATA_DIR. You already
# have the extracted clips in final_dataset/data, so leave this False unless
# you specifically want to rebuild them from the raw recordings.
RUN_CLIP_EXTRACTION = False
OVERWRITE_EXISTING_CLIPS = False   # guard: refuse to write into a non-empty DATA_DIR
MODEL_OUTPUT_DIR = None                                      # Your own previously-trained models (validation)

# --- Auto-detect MODEL_OUTPUT_DIR relative to DATA_DIR ---
if MODEL_OUTPUT_DIR is None and DATA_DIR is not None:
    _data_parent = os.path.dirname(os.path.abspath(DATA_DIR))
    for _candidate in [
        os.path.join(_data_parent, 'model_output'),
        os.path.join(_data_parent, '..', 'model_output'),
        os.path.join(_data_parent, '..', '..', 'model_output'),
        'Data/model_output',
    ]:
        _candidate = os.path.normpath(_candidate)
        if os.path.isdir(_candidate):
            MODEL_OUTPUT_DIR = _candidate
            break

# ---------------------------------------------------------------------------
# OFFICIAL PRE-TRAINED MODELS -- set paths manually here.
#
# Fill in the 3 call detectors + 1 call classifier you uploaded as a Kaggle
# dataset. Leave any of them as None to skip it (Cell 10 falls back to
# auto-discovery under /kaggle/input/*, then to Data/model_output/).
#
# Click "Add Data" in Kaggle, then use "Copy Path" on the dataset folder to
# get the exact /kaggle/input/... prefix.
#
#   Detector   192 kHz, 2 classes (noise/target)  -> fine-tunes the detector
#   Classifier 250 kHz, 15 classes                -> fine-tunes the classifier
# ---------------------------------------------------------------------------
DETECTOR_M03_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_detector/m03/ANIMAL-SPOT.pk'   # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m03/train/ANIMAL-SPOT.pk'
DETECTOR_M09_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_detector/m09/ANIMAL-SPOT.pk'    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m09/train/ANIMAL-SPOT.pk'
DETECTOR_M11_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_detector/m11/ANIMAL-SPOT.pk'    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m11/train/ANIMAL-SPOT.pk'
CLASSIFIER_M09_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_classifier/ANIMAL-SPOT.pk'  # e.g. '/kaggle/input/mymodels/batspot/models_call_classifier/m09/train/ANIMAL-SPOT.pk'

# Manual paths take priority over auto-discovery. Set to False to ignore them.
USE_MANUAL_MODEL_PATHS = True

WORKING_DIR = '/kaggle/working'
PRETRAINED_DIR = os.path.join(WORKING_DIR, 'pretrained')
os.makedirs(PRETRAINED_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Spectrogram params below are the REAL values read out of the official
# BatSpot .pk files (BatSpot_article/batspot/models_*/**/ANIMAL-SPOT.pk).
# They MUST match the model you fine-tune from, otherwise the transferred
# encoder weights were trained on a different time-frequency resolution and
# are effectively noise. Cell 10 re-verifies this against each .pk at runtime.
# ---------------------------------------------------------------------------

# --- Detector config (192 kHz, binary: noise vs target) ---
DET_CONFIG = {
    'sr': 192000,
    'num_classes': 2,
    'classes': ['noise', 'target'],
    'fmin': 1000,               # from official detector dataOpts
    'fmax': 95000,
    'n_fft': 256,
    'hop_length': 128,
    'n_freq_bins': 256,
    'sequence_len': 20,         # ms
    'max_pool': 2,
    'freq_compression': 'linear',
    # Batch 32 (official: 16). 674 clips / 32 = ~21 optimizer steps per epoch. Batch 128 gave
    # only 6 steps/epoch, which is why the old config needed absurd learning rates.
    'batch_size': 32,
    'base_lr': 1e-4,            # absolute LR (official value) — NOT scaled by batch size
    'n_epochs': 100,            # cap only: Kaggle run showed m09 still improving at the old cap of 60
                                # (best epoch 58); early_stopping_patience_epochs below stops the rest sooner
    'epochs_per_eval': 1,       # validation is cheap (clip-level, top-k windows)
    'grad_accum_steps': 1,
    'use_multi_gpu': True,
    # reference hyperparams from models_call_detector/*/config_training_gui
    'beta1': 0.5,
    'lr_decay_factor': 0.5,
    'lr_patience_epochs': 12,              # RAW epochs without improvement before halving LR
    'early_stopping_patience_epochs': 30,  # RAW epochs without improvement before stopping
}

# --- Classifier config (250 kHz, multi-species) ---
CLS_CONFIG = {
    'sr': 250000,
    'num_classes': None,        # auto-detected from your data in Cell 7
    'fmin': 10000,              # from official classifier dataOpts
    'fmax': 125000,
    'n_fft': 256,
    'hop_length': 128,
    'n_freq_bins': 256,
    'sequence_len': 20,         # ms
    'max_pool': 2,
    'freq_compression': 'linear',
    'batch_size': 32,           # ~21 steps/epoch (batch 128 = 6 steps/epoch was the real problem)
    'base_lr': 3e-4,            # measured best of {1e-4, 3e-4, 1e-3} (3 seeds each); 1e-2 is NOT needed
                                # and destabilises training (val acc collapsed to 0.42 at epoch 38)
    'n_epochs': 120,            # val score was still rising at epoch 60 with lr 3e-4
    'epochs_per_eval': 1,
    'grad_accum_steps': 1,
    'use_multi_gpu': True,
    # reference hyperparams from models_call_classifier/m09/config_train
    'beta1': 0.5,
    'lr_decay_factor': 0.5,
    'lr_patience_epochs': 15,              # RAW epochs without improvement before halving LR
    'early_stopping_patience_epochs': 40,  # RAW epochs without improvement before stopping
}

# --- Input windowing (THE main accuracy fix; see AGENTS.md "Windowing fix") ---------------
# Clips are 100 ms - 2.6 s (median ~375 ms) but every model consumes a 20 ms window.
#   'energy_crop' : TRAIN  = a fresh random crop each epoch from the loudest 20 % of the clip's windows
#                   TEST/VAL = the TEST_TOPK loudest non-overlapping windows, probabilities averaged
#   'first'       : old behaviour (only the first 20 ms of every clip) -- kept for ablation
WINDOW_MODE = 'energy_crop'
WINDOW_STRIDE = 3          # frames between candidate windows (3 frames ~ 2 ms)
TRAIN_TOP_FRAC = 0.20      # train crops come from this loudest fraction of windows
TEST_TOPK = 5              # windows averaged per clip at validation / test time

# Model selection, LR schedule and early stopping all track this validation score.
# 'balanced_accuracy' (mean per-class recall) cannot be gamed by predicting the majority class;
# the detector's 80/20 call/noise split made plain accuracy hide the all-call collapse of m03.
SELECT_METRIC = 'balanced_accuracy'   # or 'accuracy'
assert SELECT_METRIC in ('balanced_accuracy', 'accuracy'), \
    f"SELECT_METRIC must be 'balanced_accuracy' or 'accuracy', got {SELECT_METRIC!r}"

# The filename carries the recording (tape) id and there are only ~28 recordings, so a random
# clip-level split leaks recordings between train and test (Cell 7 prints a leakage check).
#   False = clip-level split, stratified by species: every species gives ~15 % to val and to test, but
#           clips of one recording land in train AND test (optimistic about new recordings).
#   True  = hold out whole recordings; RECORDING_SPLIT_SCOPE below says how.
SPLIT_BY_RECORDING = False
# Only used when SPLIT_BY_RECORDING = True.
#   'per_species' : each species is split ON ITS OWN. Its recordings go (whole) to train / val / test so that
#                   each part holds ~70/15/15 % of THAT species' clips -> every class appears in val and test.
#                   Species with 2 recordings: the bigger one trains, the smaller one is split val/test by clip.
#                   Species with 1 recording (heti): clip-level split (the only option; reported in Cell 7).
#                   CAVEAT: a recording holds several species and each is split on its own, so a test clip whose
#                   recording is in train always meets train clips of ANOTHER species there. A model that leans on
#                   the recording is scored BELOW chance (Cell 7 prints a tape-lookup baseline). Pessimistic bound.
#   'global'      : one StratifiedGroupKFold over all clips (7 folds). Recordings are indivisible, so val/test get
#                   whatever species sit in their ~4 recordings; classes can be missing, sizes are not 70/15/15.
RECORDING_SPLIT_SCOPE = 'per_species'
assert RECORDING_SPLIT_SCOPE in ('per_species', 'global'), \
    f"RECORDING_SPLIT_SCOPE must be 'per_species' or 'global', got {RECORDING_SPLIT_SCOPE!r}"

# --- Reproducibility ---
# Seeds python / numpy / torch before each model is built and trained (Cells 12-13), so a re-run
# repeats the same initialisation and sampling order. Unseeded runs of the same config spread by
# ~3-5 points (AGENTS.md 8.10); seeding makes a run repeatable but does NOT shrink that spread
# (cuDNN kernels stay non-deterministic). None = unseeded.
SEED = 42

# --- Augmentation toggle ---
# True  = time shift + gaussian noise + freq/time masking applied to train windows
# False = no extra augmentation. Random window cropping is already the augmentation that
#         matters here; the extra ones were never shown to help on ~84 clips/class.
USE_AUGMENTATION = False

# --- GPU check ---
# If DEVICE prints 'cpu', the Kaggle GPU accelerator is OFF.
# To enable: Session options (right sidebar) -> Accelerator -> GPU T4 x2 -> Save.
# The notebook must then be restarted. No code change is needed.
DEVICE = __import__('torch').device('cuda' if __import__('torch').cuda.is_available() else 'cpu')

print(f'Detector:   {DET_CONFIG["sr"]//1000}kHz, {DET_CONFIG["num_classes"]} classes '
      f'(noise/target), fmin={DET_CONFIG["fmin"]}, fmax={DET_CONFIG["fmax"]}')
print(f'Classifier: {CLS_CONFIG["sr"]//1000}kHz, {CLS_CONFIG["num_classes"]} classes (auto), '
      f'fmin={CLS_CONFIG["fmin"]}, fmax={CLS_CONFIG["fmax"]}')
print(f'Device: {DEVICE}')

# Report which manual paths resolved (a typo here is otherwise silent)
if USE_MANUAL_MODEL_PATHS:
    print('\nManual pre-trained model paths:')
    for _lbl, _p in [('detector m03', DETECTOR_M03_PATH),
                     ('detector m09', DETECTOR_M09_PATH),
                     ('detector m11', DETECTOR_M11_PATH),
                     ('classifier m09', CLASSIFIER_M09_PATH)]:
        if _p is None:
            print(f'  {_lbl:15s} (not set)')
        elif os.path.isfile(_p):
            print(f'  {_lbl:15s} OK  {_p}')
        else:
            print(f'  {_lbl:15s} !! NOT FOUND: {_p}')
else:
    print('\nManual model paths DISABLED (USE_MANUAL_MODEL_PATHS = False)')


In [ ]:
# Cell 3: Install dependencies
!pip install -q resampy soundfile tqdm

import resource
try:
    resource.setrlimit(resource.RLIMIT_NOFILE, (65535, 65535))
except Exception:
    pass

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)')


In [ ]:
# Cell 4: Imports
import os, sys, glob, json, shutil, zipfile, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import OrderedDict, defaultdict, Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import resampy
import soundfile as sf
from tqdm.auto import tqdm
from math import ceil

sys.path.insert(0, os.path.abspath('.'))

print('All imports OK')

In [ ]:
# Cell 5: Model Architecture & Loading
# --- Faithful port of animal_spot/models/ (GPL-3.0, Bergler & Schroeter) ---
# Module/attribute names MUST match the original, or saved .pk state_dicts
# will not load. In particular the residual projection is `shortcut`
# (ResidualBase assigns self.shortcut = downsample), NOT `downsample`.

def get_padding(kernel_size):
    """Return `same` padding for a given kernel size."""
    if isinstance(kernel_size, int):
        return kernel_size // 2
    return tuple(s // 2 for s in kernel_size)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, upsample=None, mid_ch=None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        if downsample is not None and upsample is not None:
            raise ValueError("Either downsample or upsample has to be None")

        if upsample is None:
            self.shortcut = downsample
            self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=3, stride=stride,
                                   padding=get_padding(3), bias=False)
        else:
            self.shortcut = upsample
            self.conv1 = nn.ConvTranspose2d(in_ch, mid_ch, kernel_size=3, stride=stride,
                                            padding=get_padding(3),
                                            output_padding=get_padding(stride), bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(mid_ch, out_ch, kernel_size=3, stride=1,
                               padding=get_padding(3), bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu2(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, upsample=None, mid_ch=None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        if downsample is not None and upsample is not None:
            raise ValueError("Either downsample or upsample has to be None")
        self.shortcut = None
        if upsample is not None or downsample is not None:
            self.shortcut = downsample if downsample is not None else upsample

        self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu1 = nn.ReLU(inplace=True)
        if upsample is not None:
            self.conv2 = nn.ConvTranspose2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                                            padding=get_padding(3),
                                            output_padding=get_padding(stride), bias=False)
        else:
            self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                                   padding=get_padding(3), bias=False)
        self.bn2 = nn.BatchNorm2d(mid_ch)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(mid_ch, out_ch * self.expansion, kernel_size=1,
                               stride=1, padding=get_padding(1), bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch * self.expansion)
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.relu2(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu3(out)
        return out


def get_block_sizes(resnet_size):
    if resnet_size == 18:
        return [2, 2, 2, 2]
    elif resnet_size == 34:
        return [3, 4, 6, 3]
    elif resnet_size in (50, 101):
        return [3, 4, 23, 3]
    elif resnet_size == 152:
        return [3, 8, 36, 3]
    raise ValueError("Unsuported resnet size: ", resnet_size)


def get_block_type(resnet_size):
    if resnet_size in (18, 34):
        return BasicBlock
    elif resnet_size in (50, 101, 152):
        return Bottleneck
    raise ValueError("Unsuported resnet size ", resnet_size)


class ResidualBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.cur_in_ch = 64

    def make_layer(self, block, out_ch, size, stride=1, shortcut="downsample"):
        layers = []
        stride_mean = stride
        if isinstance(stride, tuple):
            stride_mean = sum(stride) / len(stride)

        if shortcut == "upsample" and (stride_mean > 1
                                       or self.cur_in_ch != out_ch * block.expansion):
            shortcut = nn.ConvTranspose2d(self.cur_in_ch, out_ch * block.expansion,
                                          kernel_size=3, stride=stride,
                                          padding=get_padding(3),
                                          output_padding=get_padding(stride), bias=False)
            layers.append(block(self.cur_in_ch, out_ch, stride=stride,
                                mid_ch=self.cur_in_ch // block.expansion,
                                upsample=shortcut))
        elif shortcut == "downsample" and (stride_mean > 1
                                           or self.cur_in_ch != out_ch * block.expansion):
            shortcut = nn.Sequential(
                nn.Conv2d(self.cur_in_ch, out_ch * block.expansion, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_ch * block.expansion),
            )
            layers.append(block(self.cur_in_ch, out_ch, stride, downsample=shortcut))
        else:
            layers.append(block(self.cur_in_ch, out_ch))
        self.cur_in_ch = out_ch * block.expansion

        for _ in range(1, size):
            layers.append(block(self.cur_in_ch, out_ch))
        return nn.Sequential(*layers)


class ResidualEncoder(ResidualBase):
    def __init__(self, opts=None):
        super().__init__()
        if opts is None:
            opts = {"input_channels": 1, "conv_kernel_size": 7,
                    "max_pool": 1, "resnet_size": 18}
        self._opts = opts
        self.cur_in_ch = 64
        self.block_sizes = get_block_sizes(opts["resnet_size"])
        self.block_type = get_block_type(opts["resnet_size"])

        self.conv1 = nn.Conv2d(opts["input_channels"], out_channels=64,
                               kernel_size=opts["conv_kernel_size"], stride=(2, 2),
                               padding=get_padding(opts["conv_kernel_size"]), bias=False)
        self.bn1 = nn.BatchNorm2d(self.cur_in_ch)
        self.relu1 = nn.ReLU(inplace=True)
        if opts["max_pool"] == 1:
            self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=get_padding(3))
            stride1 = (1, 1)
        elif opts["max_pool"] == 0:
            self.max_pool = None
            stride1 = (2, 2)
        elif opts["max_pool"] == 2:
            self.max_pool = None
            stride1 = (1, 1)
        else:
            raise ValueError("Unknown max_pool option")

        self.layer1 = self.make_layer(self.block_type, 64, self.block_sizes[0], stride1)
        self.layer2 = self.make_layer(self.block_type, 128, self.block_sizes[1], (2, 2))
        self.layer3 = self.make_layer(self.block_type, 256, self.block_sizes[2], (2, 2))
        self.layer4 = self.make_layer(self.block_type, 512, self.block_sizes[3], (2, 2))

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        if self.max_pool is not None:
            x = self.max_pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def model_opts(self):
        return self._opts


class Classifier(nn.Module):
    def __init__(self, opts=None):
        super().__init__()
        if opts is None:
            opts = {"input_channels": 512, "pooling": "avg", "num_classes": 2}
        self._opts = opts
        self._layer_output = dict()
        if opts["pooling"] == "avg":
            self.pooling = lambda x: torch.mean(x, dim=-1)
        elif opts["pooling"] == "max":
            self.pooling = lambda x: torch.max(x, dim=-1)[0]
        else:
            raise ValueError("Unkown pooling option")
        self.linear = nn.Linear(opts["input_channels"], opts["num_classes"])

    def forward(self, x):
        x = x.view(x.size(0), x.size(1), -1)
        hidden_layer = self.pooling(x)
        hidden_layer = hidden_layer.view(hidden_layer.size(0), -1)
        self._layer_output["hidden_layer_1"] = hidden_layer
        output_layer = self.linear(hidden_layer)
        self._layer_output["output_layer"] = output_layer
        return output_layer

    def model_opts(self):
        return self._opts

def _assemble(encoder, classifier):
    model = nn.Sequential(OrderedDict([('encoder', encoder), ('classifier', classifier)]))
    model.eval()
    return model


def build_model(encoderOpts, num_classes, device):
    """Build a fresh model (random init) with a num_classes-wide head."""
    classifierOpts = {'input_channels': 512, 'pooling': 'avg', 'num_classes': num_classes}
    encoder = ResidualEncoder(encoderOpts)
    classifier = Classifier(classifierOpts)
    return _assemble(encoder, classifier), encoder, classifier, classifierOpts


def load_model_from_pk(path, device):
    """Load a .pk file. Returns (model, classes_dict, dataOpts)."""
    obj = torch.load(path, map_location=device, weights_only=False)
    encoder = ResidualEncoder(obj['encoderOpts'])
    encoder.load_state_dict(obj['encoderState'])
    classifier = Classifier(obj['classifierOpts'])
    classifier.load_state_dict(obj['classifierState'])
    return _assemble(encoder, classifier), obj.get('classes', {}), obj.get('dataOpts', {})


# .checkpoint files (Data/model_output/*) store no opts, so these are supplied
# from that training run's config. Verify against your own TRAIN.log if in doubt.
CHECKPOINT_ENCODER_OPTS = {'input_channels': 1, 'conv_kernel_size': 7,
                           'max_pool': 2, 'resnet_size': 18}
CHECKPOINT_DATA_OPTS = {'sr': 384000, 'preemphases': 0.98, 'n_fft': 256,
                        'hop_length': 128, 'n_freq_bins': 256, 'fmin': 18000,
                        'fmax': 90000, 'freq_compression': 'linear',
                        'min_level_db': -100, 'ref_level_db': 20}


def load_any_model(path, device):
    """Load either a .pk or a .checkpoint file."""
    obj = torch.load(path, map_location=device, weights_only=False)
    if 'encoderOpts' in obj:
        return load_model_from_pk(path, device)
    if 'modelState' in obj:
        classes = obj.get('classes', {})
        classifierOpts = {'input_channels': 512, 'pooling': 'avg',
                          'num_classes': len(classes)}
        encoder = ResidualEncoder(CHECKPOINT_ENCODER_OPTS)
        classifier = Classifier(classifierOpts)
        ms = obj['modelState']
        enc_state = {k[len('encoder.'):]: v for k, v in ms.items() if k.startswith('encoder.')}
        cls_state = {k[len('classifier.'):]: v for k, v in ms.items() if k.startswith('classifier.')}
        encoder.load_state_dict(enc_state)
        classifier.load_state_dict(cls_state)
        return _assemble(encoder, classifier), classes, dict(CHECKPOINT_DATA_OPTS)
    raise ValueError(f'Unknown model format: {list(obj.keys())}')


print('Architecture + loading functions defined')


In [ ]:
# Cell 6: Transforms & Dataset
#
# WHY THIS CELL CHANGED (see AGENTS.md "Windowing fix"):
# The clips are 100 ms - 2.6 s long (median ~375 ms) but every model sees a
# 20 ms window. The previous version kept spec[:seq_len] -- the FIRST 20 ms of
# every clip, always the same one -- so ~95 % of each clip was never used and
# the network memorised 674 fixed inputs. The official ANIMAL-SPOT code
# trains on RANDOM 20 ms crops and predicts with a sliding window; this cell
# restores that behaviour (WINDOW_MODE = 'first' reproduces the old one).
#
# Front end = official animal_spot/data/transforms.py, step for step:
#   mono mean -> resample (kaiser_best) -> pre-emphasis 0.98 ->
#   torch.stft(center=False) / sqrt(sum(w^2)) -> power -> keep fmin..fmax bins
#   -> NEAREST interpolation to n_freq_bins -> 10*log10 floored at -100 dB
#   -> per-window min-max normalisation (the official models were trained with
#      min_max_norm=true, and predict.py normalises each window on its own).
# The old front end used scipy.ndimage.zoom (cubic spline) on the *power*
# spectrum, which rings negative and is clipped to the -100 dB floor.

import random as _random
import hashlib


def get_class_from_filename(filename):
    """Extract class label from BatSpot filename (before first '-')."""
    return os.path.basename(filename).split('-', 1)[0]


def load_audio_file(path, target_sr):
    """Load audio (mono mean, like the official loader) and resample to target_sr."""
    audio, native_sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if native_sr != target_sr:
        audio = resampy.resample(audio, native_sr, target_sr, filter='kaiser_best')
    return audio.astype(np.float32), target_sr


def clip_to_db_spectrogram(audio, cfg, preemphasis=0.98, min_level_db=-100):
    """Official front end up to (and incl.) amplitude->dB. Returns (frames, n_freq_bins) float32.
    NOT normalised: min-max is applied per 20 ms window later."""
    y = torch.from_numpy(audio)[None]
    y = torch.cat((y[:, :1], y[:, 1:] - preemphasis * y[:, :-1]), dim=-1)
    win = torch.hann_window(cfg['n_fft'])
    S = torch.stft(y, n_fft=cfg['n_fft'], hop_length=cfg['hop_length'], window=win,
                   center=False, onesided=True, return_complex=True)
    S = torch.view_as_real(S).transpose(1, 2)            # (1, frames, bins, 2)
    S = S / win.pow(2).sum().sqrt()
    S = S.pow(2).sum(-1)                                  # power, (1, frames, n_fft/2+1)
    n_fft = (S.size(2) - 1) * 2
    lo = int(max(0, np.floor(n_fft * cfg['fmin'] / cfg['sr'])))
    hi = int(min(n_fft - 1, np.ceil(n_fft * cfg['fmax'] / cfg['sr'])))
    S = S[:, :, lo:hi]
    S = torch.nn.functional.interpolate(S[None], size=(S.size(1), cfg['n_freq_bins']),
                                        mode='nearest')[0]
    S = torch.clamp(S, min=float(np.exp(min_level_db / 10 * np.log(10))))
    return (10 * torch.log10(S))[0].numpy().astype(np.float32)


def minmax_normalize(spec):
    """Min-max normalise to [0, 1] (official MinMaxNormalize)."""
    spec = spec - spec.min()
    m = spec.max()
    return spec / m if m > 0 else spec


def pad_window(win, seq_len):
    """Centre-pad a short clip with zeros to seq_len frames (official PaddedSubsequenceSampler)."""
    n = win.shape[0]
    if n >= seq_len:
        return win
    out = np.zeros((seq_len, win.shape[1]), dtype=np.float32)
    p = (seq_len - n) // 2
    out[p:p + n] = win
    return out


def window_scores(db, seq_len, stride):
    """Candidate window starts + a loudness score (mean of the per-frame peak dB)."""
    n = db.shape[0]
    if n <= seq_len:
        return np.array([0]), np.array([0.0])
    peak = db.max(axis=1)
    cs = np.concatenate([[0.0], np.cumsum(peak)])
    starts = np.arange(0, n - seq_len + 1, stride)
    return starts, (cs[starts + seq_len] - cs[starts]) / seq_len


def train_window_starts(starts, scores, mode, top_frac):
    """Windows a training clip may be cropped at: the loudest `top_frac` (>= 3).
    Selected by RANK, not by a percentile threshold: with tied scores (flat / saturated clips)
    a threshold keeps every tied window, i.e. a uniform crop over the whole clip."""
    if mode == 'first':
        return starts[:1]
    n_keep = min(len(starts), max(3, int(np.ceil(top_frac * len(starts)))))
    return starts[np.argsort(-scores, kind='stable')[:n_keep]]


def eval_window_starts(starts, scores, mode, k, seq_len):
    """Top-k loudest windows that overlap by less than half a window (greedy NMS)."""
    if mode == 'first':
        return starts[:1]
    chosen = []
    for i in np.argsort(-scores):
        if all(abs(starts[i] - starts[j]) >= seq_len // 2 for j in chosen):
            chosen.append(i)
        if len(chosen) >= k:
            break
    return starts[chosen]


def _augment_spec(spec):
    """Optional on-the-fly spectrogram augmentation (training only, USE_AUGMENTATION).
    Input shape (seq_len, n_freq_bins). Time shift, Gaussian noise, SpecAugment masks.
    NOTE: random window cropping already provides most of the useful augmentation;
    this stays OFF by default (small dataset)."""
    seq_len, n_freq = spec.shape
    spec = spec.copy()
    shift = int(np.random.uniform(-0.20, 0.20) * seq_len)
    if shift > 0:
        spec = np.concatenate([np.zeros((shift, n_freq)), spec[:-shift]], axis=0)
    elif shift < 0:
        spec = np.concatenate([spec[-shift:], np.zeros((-shift, n_freq))], axis=0)
    spec = np.clip(spec + np.random.normal(0, 0.01, spec.shape), 0.0, 1.0)
    f_mask = int(np.random.uniform(0, 0.20) * n_freq)
    if f_mask > 0:
        f0 = np.random.randint(0, max(1, n_freq - f_mask))
        spec[:, f0:f0 + f_mask] = 0.0
    t_mask = int(np.random.uniform(0, 0.20) * seq_len)
    if t_mask > 0:
        t0 = np.random.randint(0, max(1, seq_len - t_mask))
        spec[t0:t0 + t_mask, :] = 0.0
    return spec.astype(np.float32)


class WindowedBatDataset(Dataset):
    """Clip-level dataset served as 20 ms windows.

    train=True : __getitem__(i) -> (window, label); a RANDOM window from the
                 loudest TRAIN_TOP_FRAC of clip i (new crop every epoch).
    train=False: flat list of the TEST_TOPK loudest non-overlapping windows of
                 every clip; __getitem__ -> (window, label, clip_index).
                 Use predict_proba() to average the windows back to clip level.

    The full-clip dB spectrogram is cached once to disk (np.save) and memory-
    mapped, so a window read costs microseconds.
    """
    _CACHE_ROOT = os.path.join(WORKING_DIR, 'spec_cache')
    _CACHE_VERSION = 'v2'        # bump when the front end changes (invalidates old caches)

    def __init__(self, file_names, class_to_idx, cfg, train=False, augment=False,
                 mode=None, topk=None):
        self.file_names = list(file_names)
        self.class_to_idx = class_to_idx
        self.cfg = cfg
        self.train = train
        self.augment = augment and train
        self.mode = mode or WINDOW_MODE
        self.topk = topk or TEST_TOPK
        self.seq_len = int(float(cfg['sequence_len']) / 1000 * cfg['sr'] / cfg['hop_length'])
        key = (f"{self._CACHE_VERSION}_{cfg['sr']}_{cfg['n_fft']}_{cfg['hop_length']}_"
               f"{cfg['n_freq_bins']}_{cfg['fmin']}_{cfg['fmax']}")
        self.cache_dir = os.path.join(self._CACHE_ROOT, hashlib.md5(key.encode()).hexdigest()[:8])
        os.makedirs(self.cache_dir, exist_ok=True)
        self._build_cache()
        self.labels = np.array([class_to_idx[get_class_from_filename(f)] for f in self.file_names])
        # window bookkeeping
        self.train_starts, self.items = [], []
        for i, f in enumerate(self.file_names):
            db = np.load(self._spec_path(f), mmap_mode='r')
            starts, scores = window_scores(db, self.seq_len, WINDOW_STRIDE)
            if train:
                self.train_starts.append(train_window_starts(starts, scores, self.mode, TRAIN_TOP_FRAC))
            else:
                self.items += [(i, int(s)) for s in
                               eval_window_starts(starts, scores, self.mode, self.topk, self.seq_len)]

    def _spec_path(self, f):
        return os.path.join(self.cache_dir, os.path.basename(f) + '.npy')

    def _build_cache(self):
        missing = [f for f in self.file_names if not os.path.exists(self._spec_path(f))]
        if not missing:
            return
        print(f'[WindowedBatDataset] caching {len(missing)} clips -> {self.cache_dir}')
        for f in tqdm(missing, desc='cache', leave=False):
            audio, _ = load_audio_file(f, self.cfg['sr'])
            np.save(self._spec_path(f), clip_to_db_spectrogram(audio, self.cfg))

    def _window(self, i, start):
        db = np.load(self._spec_path(self.file_names[i]), mmap_mode='r')
        win = np.array(db[start:start + self.seq_len])
        # normalise FIRST, pad second (official order): padding a dB array with 0.0 before
        # min-max makes the padding the array maximum and squashes the real signal.
        return pad_window(minmax_normalize(win), self.seq_len).astype(np.float32)

    def __len__(self):
        return len(self.file_names) if self.train else len(self.items)

    def __getitem__(self, idx):
        if self.train:
            win = self._window(idx, int(_random.choice(self.train_starts[idx])))
            if self.augment:
                win = _augment_spec(win)
            return torch.from_numpy(win).unsqueeze(0), int(self.labels[idx])
        i, start = self.items[idx]
        return torch.from_numpy(self._window(i, start)).unsqueeze(0), int(self.labels[i]), i


def predict_proba(model, dataset, device, batch_size=256):
    """Clip-level probabilities: softmax averaged over each clip's windows.
    `dataset` must be a train=False WindowedBatDataset. Returns (probs[n_clips, C], labels[n_clips])."""
    assert not dataset.train, 'predict_proba needs an evaluation (train=False) dataset'
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    sums, counts = None, np.zeros(len(dataset.file_names))
    with torch.no_grad():
        for inputs, _, clip_idx in loader:
            p = torch.softmax(model(inputs.to(device)).float(), dim=1).cpu().numpy()
            if sums is None:
                sums = np.zeros((len(dataset.file_names), p.shape[1]))
            for row, ci in zip(p, clip_idx.numpy()):
                sums[ci] += row
                counts[ci] += 1
    if not (counts > 0).all():   # a clip without windows would silently score as class 0
        _bad = [os.path.basename(dataset.file_names[i]) for i in np.flatnonzero(counts == 0)][:5]
        raise RuntimeError(f'{int((counts == 0).sum())} clip(s) produced no windows, e.g. {_bad}')
    return sums / counts[:, None], dataset.labels.copy()


print('Windowed transforms + WindowedBatDataset defined '
      f'(WINDOW_MODE={WINDOW_MODE!r}, train top {TRAIN_TOP_FRAC:.0%} loudest windows, test top-{TEST_TOPK})')

In [ ]:
# Cell 7: Data Discovery

# Find all wav files
all_wavs = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True))
if not all_wavs:
    # Try flat structure
    all_wavs = sorted(glob.glob(os.path.join(DATA_DIR, '*.wav')))

print(f'Found {len(all_wavs)} wav files')

# Auto-detect classes from folder names
class_counts = Counter()
for w in all_wavs:
    cls = get_class_from_filename(w)
    class_counts[cls] += 1

all_classes = sorted(class_counts.keys())
print(f'Classes ({len(all_classes)}):')
for c in all_classes:
    print(f'  {c}: {class_counts[c]} files')

# Update classifier config with actual num_classes
CLS_CONFIG['num_classes'] = len(all_classes)
print(f'\nClassifier num_classes: {CLS_CONFIG["num_classes"]}')

# Detector class mapping: noise -> 0, all bat species -> 1 (call)
DET_CLASSES = ['noise', 'call']
DET_CLASS_TO_IDX = {'noise': 0, 'call': 1}

# Classifier class mapping: sorted species classes (incl. noise)
CLS_CLASSES = all_classes
CLS_CLASS_TO_IDX = {c: i for i, c in enumerate(all_classes)}
CLS_IDX_TO_CLASS = {i: c for c, i in CLS_CLASS_TO_IDX.items()}

print(f'\nDetector mapping: {DET_CLASS_TO_IDX}')
print(f'Classifier mapping: {CLS_CLASS_TO_IDX}')

# ---------------------------------------------------------------------------
# DATA SPLIT: train / val / test = 70 / 15 / 15, ONE shared split for detector and classifier
# so the cascade has no leakage. Three modes (Cell 2):
#
#   SPLIT_BY_RECORDING = False
#       clip-level, stratified by species (incl. noise): every species gives ~15 % to val and to test.
#
#   SPLIT_BY_RECORDING = True, RECORDING_SPLIT_SCOPE = 'per_species'   (default for True)
#       each species is split ON ITS OWN, and whole recordings are the unit: a species' recordings go to
#       train / val / test so that each part holds ~70 / 15 / 15 % of THAT species' clips. Every class
#       therefore appears in val and test, and no recording contributes the same species to train and
#       to val/test (species with < 3 recordings cannot be split this way; see the rules below).
#
#   SPLIT_BY_RECORDING = True, RECORDING_SPLIT_SCOPE = 'global'
#       one StratifiedGroupKFold over all clips: recordings are indivisible and shared between species,
#       so val/test get whatever species happen to sit in their ~4 recordings (classes go missing).
#
# RECORDING LEAKAGE WARNING: the filename carries the tape (recording) id
# (CLASS-LABEL_ID_YEAR_TAPE_START_END.wav). This dataset has only ~28 recordings (heti is a
# single one), so a clip-level split puts clips of the SAME recording in train and test.
# The network can then score well by recognising the recording (background, mic distance)
# rather than the species -> optimistic numbers. The check at the end of this cell quantifies it.
# ---------------------------------------------------------------------------
import re
import itertools
# Recording timestamp YYYYMMDD + HHMMSS. The dataset clips write 20260529-202000; Cell 8's extractor
# (and AudioMoth file names) write 20260529_202000, so accept either separator.
_TAPE_RE = re.compile(r'(\d{8})[-_](\d{6})')

def recording_id(path):
    """Recording (tape) id = the YYYYMMDD-HHMMSS timestamp inside the file name. Searching for it
    (instead of taking field 4) works for CLASS-LABEL_ID_YEAR_TAPE_START_END.wav and for other
    layouts such as Cell 8's extractor output. Fallback: the file stem (= no grouping)."""
    stem = os.path.basename(path)[:-4]
    m = _TAPE_RE.search(stem)
    return f'{m.group(1)}-{m.group(2)}' if m else stem

SPLIT_FRACTIONS = (0.70, 0.15, 0.15)          # train / val / test


def _assign_recordings(sizes, fractions, rng):
    """Give every recording (an indivisible group of clips) to one of len(fractions) parts so that each
    part's clip total is as close as possible to fraction * total, every part getting >= 1 recording.
    Up to 11 recordings: exhaustive search, i.e. the optimum (ties are broken by `rng`).
    More: largest-first greedy into the part furthest below its target."""
    sizes = np.asarray(sizes, dtype=float)
    k, p = len(sizes), len(fractions)
    target = np.asarray(fractions, dtype=float) * sizes.sum()
    if k <= 11:
        allp = np.array(list(itertools.product(range(p), repeat=k)))             # (p**k, k)
        tot = np.stack([(allp == j) @ sizes for j in range(p)], axis=1)          # clips per part
        cost = ((tot - target) ** 2).sum(axis=1)
        cost[~np.all([(allp == j).any(axis=1) for j in range(p)], axis=0)] = np.inf   # no empty part
        return allp[rng.choice(np.flatnonzero(cost <= cost.min() + 1e-9))]
    parts, tot = np.full(k, -1), np.zeros(p)
    for i in np.argsort(-sizes, kind='stable'):
        j = int(np.argmax(target - tot))
        parts[i] = j
        tot[j] += sizes[i]
    for j in range(p):                                                             # repair: no empty part
        if not (parts == j).any():
            cand = np.flatnonzero(parts == np.bincount(parts, minlength=p).argmax())
            parts[cand[np.argmin(sizes[cand])]] = j
    return parts


def split_by_recording_per_species(files, species, rec_ids, fractions=SPLIT_FRACTIONS, seed=42):
    """SPLIT_BY_RECORDING, one species at a time. Returns (train, val, test, rules); `rules[species]`
    = (number of recordings, rule applied). Every file lands in exactly one part.
      >= 3 recordings : whole recordings -> train / val / test, ~fractions of that species' clips
      == 2 recordings : the bigger recording -> train; the smaller one is split val/test by clip
                        (train is still recording-disjoint from val/test; val and test share a tape)
      == 1 recording  : clip-level split (the only option; train, val and test share the recording)"""
    rng = np.random.default_rng(seed)
    parts, rules = [[], [], []], {}
    for sp in sorted(set(species)):
        by_rec = {}
        for i, (s, r) in enumerate(zip(species, rec_ids)):
            if s == sp:
                by_rec.setdefault(r, []).append(i)
        recs = sorted(by_rec)
        n, k = sum(len(v) for v in by_rec.values()), len(recs)
        if k >= 3:
            rules[sp] = (k, 'whole recordings held out')
            for r, j in zip(recs, _assign_recordings([len(by_rec[r]) for r in recs], fractions, rng)):
                parts[j] += by_rec[r]
        elif k == 2:
            rules[sp] = (k, 'bigger recording -> train; smaller one split val/test by clip')
            big, small = sorted(recs, key=lambda r: (-len(by_rec[r]), r))
            parts[0] += by_rec[big]
            idx = list(by_rec[small])
            rng.shuffle(idx)
            parts[1] += idx[:(len(idx) + 1) // 2]
            parts[2] += idx[(len(idx) + 1) // 2:]
        else:
            idx = [i for r in recs for i in by_rec[r]]
            if n < 3:
                rules[sp] = (k, 'LESS THAN 3 CLIPS -> all in train (cannot be evaluated)')
                parts[0] += idx
                continue
            rules[sp] = (k, 'CLIP-LEVEL (single recording: shared by train/val/test)')
            rng.shuffle(idx)
            n_val, n_test = max(1, round(fractions[1] * n)), max(1, round(fractions[2] * n))
            parts[1] += idx[:n_val]
            parts[2] += idx[n_val:n_val + n_test]
            parts[0] += idx[n_val + n_test:]
    return tuple([files[i] for i in sorted(p)] for p in parts) + (rules,)


all_species_labels = [get_class_from_filename(w) for w in all_wavs]
all_rec_ids = [recording_id(w) for w in all_wavs]
_rec_of = dict(zip(all_wavs, all_rec_ids))
all_rec_groups = set(all_rec_ids)

if SPLIT_BY_RECORDING and RECORDING_SPLIT_SCOPE == 'per_species':
    train_wavs, val_wavs, test_wavs, _split_rules = split_by_recording_per_species(
        all_wavs, all_species_labels, all_rec_ids, SPLIT_FRACTIONS, seed=42)
    train_labels = [get_class_from_filename(w) for w in train_wavs]
    val_labels = [get_class_from_filename(w) for w in val_wavs]
    test_labels = [get_class_from_filename(w) for w in test_wavs]
    print('SPLIT_BY_RECORDING = True, scope per_species: every species is split on its own, whole '
          f'recordings per part, target {SPLIT_FRACTIONS[0]:.0%}/{SPLIT_FRACTIONS[1]:.0%}/'
          f'{SPLIT_FRACTIONS[2]:.0%} of THAT species\' clips.')
    _tr_c, _va_c, _te_c = Counter(train_labels), Counter(val_labels), Counter(test_labels)
    print(f'\n  {"species":<8}{"clips":>6}{"recs":>5} | {"train":>5}{"val":>5}{"test":>5} | '
          f'{"train%":>7}{"val%":>6}{"test%":>6} | rule')
    for _sp in all_classes:
        _n = class_counts[_sp]
        _k, _rule = _split_rules[_sp]
        _c = (_tr_c[_sp], _va_c[_sp], _te_c[_sp])
        print(f'  {_sp:<8}{_n:>6}{_k:>5} | {_c[0]:>5}{_c[1]:>5}{_c[2]:>5} | '
              f'{100 * _c[0] / _n:>7.1f}{100 * _c[1] / _n:>6.1f}{100 * _c[2] / _n:>6.1f} | {_rule}')
elif SPLIT_BY_RECORDING:        # RECORDING_SPLIT_SCOPE == 'global'
    from sklearn.model_selection import StratifiedGroupKFold
    _folds = [te for _, te in StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=1)
              .split(all_wavs, all_species_labels, all_rec_ids)]
    test_wavs = [all_wavs[i] for i in _folds[0]]
    val_wavs = [all_wavs[i] for i in _folds[1]]
    _held = set(_folds[0]) | set(_folds[1])
    train_wavs = [w for i, w in enumerate(all_wavs) if i not in _held]
    train_labels = [get_class_from_filename(w) for w in train_wavs]
    val_labels = [get_class_from_filename(w) for w in val_wavs]
    test_labels = [get_class_from_filename(w) for w in test_wavs]
    print('SPLIT_BY_RECORDING = True, scope global: whole recordings are held out as ONE pool. This is NOT '
          '70/15/15 and classes with few recordings will be missing from val/test -- read per-class '
          'numbers with care (RECORDING_SPLIT_SCOPE = "per_species" splits each species on its own).')
else:
    train_wavs, test_wavs, train_labels, test_labels = train_test_split(
        all_wavs, all_species_labels, test_size=0.15, random_state=42,
        stratify=all_species_labels)
    train_wavs, val_wavs, train_labels, val_labels = train_test_split(
        train_wavs, train_labels, test_size=0.15/0.85, random_state=42,
        stratify=train_labels)

print(f'\nSplit: train={len(train_wavs)}, val={len(val_wavs)}, test={len(test_wavs)}')
print('Train class dist:', Counter(train_labels))
print('Val class dist:  ', Counter(val_labels))
print('Test class dist: ', Counter(test_labels))
_missing = {n: [c for c in all_classes if c not in d] for n, d in
            (('train', Counter(train_labels)), ('val', Counter(val_labels)), ('test', Counter(test_labels)))}
for _n, _m in _missing.items():
    if _m:
        print(f'  WARNING: {_n} has NO clips of {_m} -- those classes cannot be scored/selected on there.')

# --- recording leakage -----------------------------------------------------------------
_train_rec = {_rec_of[w] for w in train_wavs}
_train_rec_sp = {(_rec_of[w], get_class_from_filename(w)) for w in train_wavs}
_n_leak = sum(_rec_of[w] in _train_rec for w in test_wavs)
print(f'\nLEAKAGE CHECK: {_n_leak}/{len(test_wavs)} test clips come from a recording that also '
      f'appears in train ({len(all_rec_groups)} recordings in total).')
_same_va = sum((_rec_of[w], get_class_from_filename(w)) in _train_rec_sp for w in val_wavs)
_same_te = sum((_rec_of[w], get_class_from_filename(w)) in _train_rec_sp for w in test_wavs)
print(f'  same SPECIES and same recording as a train clip: val {_same_va}/{len(val_wavs)}, '
      f'test {_same_te}/{len(test_wavs)}  (the shortcut "recognise the tape -> name the species")')
_leaky = Counter(get_class_from_filename(w) for w in test_wavs
                 if (_rec_of[w], get_class_from_filename(w)) in _train_rec_sp)
if _leaky:
    print('  test clips per species that still share a recording with train:', dict(sorted(_leaky.items())))
if len(all_rec_groups) == len(all_wavs):
    print('  WARNING: every clip is its own recording -- recording_id() found no tape id in the '
          'file names, so this check and SPLIT_BY_RECORDING are meaningless for this dataset.')


def _tape_lookup_baseline(test_files, label_of):
    """No audio, no model: label every test clip like the MAJORITY of the train clips from the same recording.
    Returns (clips whose recording is in train, accuracy, balanced accuracy) or None."""
    by = {}
    for w in train_wavs:
        by.setdefault(_rec_of[w], Counter())[label_of(w)] += 1
    ys = [label_of(w) for w in test_files if _rec_of[w] in by]
    ps = [by[_rec_of[w]].most_common(1)[0][0] for w in test_files if _rec_of[w] in by]
    if not ys:
        return None
    bal = float(np.mean([np.mean([p == c for y, p in zip(ys, ps) if y == c]) for c in sorted(set(ys))]))
    return len(ys), float(np.mean([y == p for y, p in zip(ys, ps)])), bal


print('\nTAPE-LOOKUP BASELINE (predict a test clip like the majority of train clips from its own recording):')
for _task, _label_of, _chance in (('species      ', get_class_from_filename, f'1/{len(all_classes)}'),
                                  ('noise-vs-call', lambda w: 'noise' if get_class_from_filename(w) == 'noise'
                                   else 'call', '1/2')):
    _r = _tape_lookup_baseline(test_wavs, _label_of)
    if _r is None:
        print(f'  {_task}: no test clip shares a recording with train (the recording tells nothing)')
    else:
        print(f'  {_task}: {_r[0]}/{len(test_wavs)} test clips have their recording in train -> '
              f'accuracy {_r[1]:.3f}, balanced accuracy {_r[2]:.3f}   (chance {_chance})')
print('  Reading: ~chance = the recording says nothing (a fair estimate for NEW recordings); far above = the split '
      'leaks the recording; far below = the recording points at ANOTHER label (adversarial).')
if SPLIT_BY_RECORDING and RECORDING_SPLIT_SCOPE == 'per_species':
    print('  NOTE per_species: every species is split on its own, so a test clip whose recording is in train always '
          'meets train clips of ANOTHER species there. A model that leans on the recording (background, mic) is '
          'then scored BELOW chance. Read this split as a pessimistic bound; the "same SPECIES" line above shows '
          'the own-species leakage is 0 wherever a species had >= 2 recordings.')
elif _n_leak:
    print('  -> test scores are optimistic about NEW recordings; set SPLIT_BY_RECORDING = True '
          'for a stricter (but noisier) estimate.')


In [ ]:
# Cell 8: Clip Extraction (Python port of export_clips.R)

def extract_clips_from_selections(raw_audio_dir, selections_dir, output_dir, target_sr=384000):
    """
    Extract audio clips from raw recordings using Raven selection tables.
    Python port of Data/scripts/export_clips.R.
    """
    os.makedirs(output_dir, exist_ok=True)
    extracted = 0

    # Iterate over species folders in selections_dir
    for species in sorted(os.listdir(selections_dir)):
        species_dir = os.path.join(selections_dir, species)
        if not os.path.isdir(species_dir):
            continue

        for sel_file in sorted(glob.glob(os.path.join(species_dir, '*.txt'))):
            # Read Raven selection table (tab-separated)
            with open(sel_file, 'r') as f:
                lines = f.readlines()
            if len(lines) < 2:
                continue

            # Parse header
            header = lines[0].strip().split('\t')
            try:
                begin_idx = header.index('Begin Time (s)')
                end_idx = header.index('End Time (s)')
                view_idx = header.index('View')
            except ValueError:
                continue

            # Find matching raw audio file by date/time in selection table filename
            # Selection filename format: YYYYMMDD_HHMMSS_species.txt
            sel_name = os.path.basename(sel_file).replace('.txt', '')
            date_time = '_'.join(sel_name.split('_')[:2])
            matching_audio = None
            for audio_file in glob.glob(os.path.join(raw_audio_dir, '*.WAV')):
                audio_name = os.path.basename(audio_file).replace('.WAV', '')
                if date_time in audio_name:
                    matching_audio = audio_file
                    break
            if matching_audio is None:
                continue

            # Load audio
            audio, native_sr = sf.read(matching_audio, dtype='float32')
            if audio.ndim > 1:
                audio = audio[:, 0]

            # Extract each selection
            for line in lines[1:]:
                parts = line.strip().split('\t')
                if len(parts) <= max(begin_idx, end_idx, view_idx):
                    continue
                view = parts[view_idx]
                if 'Spectrogram' not in view:
                    continue
                begin_time = float(parts[begin_idx])
                end_time = float(parts[end_idx])
                start_sample = int(begin_time * native_sr)
                end_sample = int(end_time * native_sr)
                clip = audio[start_sample:end_sample]
                if len(clip) == 0:
                    continue
                start_ms = int(begin_time * 1000)
                end_ms = int(end_time * 1000)
                out_name = f'{species}-bat_{date_time}_{sel_name}_{start_ms}_{end_ms}.wav'
                out_path = os.path.join(output_dir, out_name)
                sf.write(out_path, clip, native_sr)
                extracted += 1

    print(f'Extracted {extracted} clips to {output_dir}')
    return extracted

# Extraction is opt-in, and refuses to clobber an existing dataset.
if not RUN_CLIP_EXTRACTION:
    print('Clip extraction SKIPPED (RUN_CLIP_EXTRACTION = False in Cell 2).')
    print(f'Using the clips already present in: {DATA_DIR}')
elif not (RAW_AUDIO_DIR and SELECTIONS_DIR):
    print('Clip extraction skipped: set RAW_AUDIO_DIR and SELECTIONS_DIR in Cell 2.')
else:
    _existing = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)
    if _existing and not OVERWRITE_EXISTING_CLIPS:
        print(f'Clip extraction ABORTED: {len(_existing)} wav files already exist in')
        print(f'  {DATA_DIR}')
        print('Re-extracting would overwrite them. Set OVERWRITE_EXISTING_CLIPS = True')
        print('in Cell 2 to force it, or point DATA_DIR at an empty folder.')
    else:
        if _existing:
            print(f'WARNING: overwriting {len(_existing)} existing clips in {DATA_DIR}')
        extract_clips_from_selections(RAW_AUDIO_DIR, SELECTIONS_DIR, DATA_DIR)

In [ ]:
# Cell 9: Training Infrastructure

def set_seed(seed):
    """Seed python / numpy / torch (incl. CUDA); DataLoader workers derive their seeds from torch."""
    if seed is None:
        return
    _random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def compute_class_weights(file_names, class_to_idx, num_labels=None):
    """Inverse-frequency class weights, one per MODEL OUTPUT class.

    num_labels must be the classifier head size (config['num_classes']), not
    len(class_to_idx): the detector folds 8 bat species + 'noise' into 2
    labels, so keying the vector off the dict length yields a 9-element
    tensor for a 2-output head and CrossEntropyLoss rejects it.
    """
    labels = [class_to_idx[get_class_from_filename(f)] for f in file_names]
    counts = Counter(labels)
    total = len(labels)
    n = num_labels if num_labels else len(class_to_idx)
    weights = torch.zeros(n)
    n_present = len(counts)
    for idx, count in counts.items():
        weights[idx] = total / (n_present * count) if count else 0.0
    return weights

def make_weighted_sampler(file_names, class_to_idx, num_labels=None):
    """Create WeightedRandomSampler for class balancing."""
    labels = [class_to_idx[get_class_from_filename(f)] for f in file_names]
    weights = compute_class_weights(file_names, class_to_idx, num_labels)
    sample_weights = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

def train_model(model, train_dataset, val_dataset, config, device, save_path=None):
    """
    Fine-tune `model` on windowed clips. AMP, optional gradient accumulation, DataParallel.

    train_dataset : WindowedBatDataset(train=True)  -> a new random 20 ms crop per clip per epoch
    val_dataset   : WindowedBatDataset(train=False) -> clip-level score = softmax averaged over
                    that clip's top-k windows (same protocol used for the test set)
    Model selection / LR schedule / early stopping all use SELECT_METRIC on the validation
    set (balanced accuracy by default: plain accuracy cannot tell a real detector from an
    "always predict call" one on an 80/20 split).
    save_path : if given, the training curves (train loss, validation accuracy / balanced
    accuracy, learning rate) are written there as a PNG. It never affects which weights are
    returned (the best-validation weights are always restored).
    Returns (model, history, best_score).
    """
    # Weighted sampler for balanced batches (50/50 per class).
    # NOTE: We do NOT also pass class_weights to CrossEntropyLoss.
    # WeightedRandomSampler already balances; adding loss weights on top
    # creates a 4x noise penalty that collapses m03 (weak pretrain) to all-noise.
    all_file_names = train_dataset.file_names
    class_to_idx = train_dataset.class_to_idx
    sampler = make_weighted_sampler(all_file_names, class_to_idx,
                                   config.get('num_classes'))

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                              sampler=sampler, num_workers=4, pin_memory=True,
                              persistent_workers=True)

    # Loss, optimizer, scheduler
    loss_fn = nn.CrossEntropyLoss()  # unweighted; sampler already balances batches
    effective_lr = config['base_lr']  # paper LR is absolute; do NOT scale by batch size
    optimizer = optim.Adam(model.parameters(), lr=effective_lr, betas=(config.get('beta1', 0.5), 0.999))
    patience_lr = int(max(1, ceil(config.get('lr_patience_epochs', 8) / config['epochs_per_eval'])))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=patience_lr, factor=config.get('lr_decay_factor', 0.5), threshold=1e-3, threshold_mode='abs')
    patience_early = int(config.get('early_stopping_patience_epochs', 20))  # RAW epochs, read from config
    grad_accum_steps = config.get('grad_accum_steps', 1)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    # Multi-GPU
    is_dp = config.get('use_multi_gpu', False) and torch.cuda.device_count() > 1
    if is_dp:
        model = nn.DataParallel(model)
        print(f'DataParallel across {torch.cuda.device_count()} GPUs')

    print(f'Grad accum: {grad_accum_steps}, AMP: {use_amp}, DP: {is_dp}, '
          f'batch: {config["batch_size"]} ({len(train_loader)} steps/epoch)')
    print(f'Effective LR: {effective_lr:.2e}   selection metric: {SELECT_METRIC}')
    print(f'Patience LR: {patience_lr} validations, Patience ES: {patience_early} raw epochs')

    history = {'train_loss': [], 'val_acc': [], 'val_bal_acc': [], 'val_epoch': [], 'lr': [],
               'best_epoch': None}
    best_score = float('-inf')   # so a first validation score of exactly 0.0 is still captured
    best_state = None
    no_improve = 0

    for epoch in range(config['n_epochs']):
        # Train
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["n_epochs"]}', leave=False)
        for step, (inputs, labels) in enumerate(pbar):
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.amp.autocast(device.type, enabled=use_amp):
                output = model(inputs)
                loss = loss_fn(output, labels) / grad_accum_steps
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            epoch_loss += loss.item() * grad_accum_steps
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        history['train_loss'].append(avg_loss)

        # Validate every epochs_per_eval
        if (epoch + 1) % config['epochs_per_eval'] == 0 or epoch == config['n_epochs'] - 1:
            val_probs, val_labels = predict_proba(model, val_dataset, device)
            val_pred = val_probs.argmax(axis=1)
            val_acc = accuracy_score(val_labels, val_pred)
            val_bal = balanced_accuracy_score(val_labels, val_pred)
            val_score = {'balanced_accuracy': val_bal, 'accuracy': val_acc}[SELECT_METRIC]  # KeyError on a typo
            history['val_acc'].append(val_acc)
            history['val_bal_acc'].append(val_bal)
            history['val_epoch'].append(epoch + 1)
            history['lr'].append(optimizer.param_groups[0]['lr'])
            scheduler.step(val_score)

            if val_score > best_score:
                best_score = val_score
                history['best_epoch'] = epoch + 1
                no_improve = 0
                # Always snapshot bare keys (strip DataParallel's "module." prefix)
                _sd = model.module.state_dict() if is_dp else model.state_dict()
                best_state = {k: v.detach().clone() for k, v in _sd.items()}
            else:
                no_improve += config['epochs_per_eval']  # RAW epochs elapsed, not validation steps

            print(f'  Epoch {epoch+1}: loss={avg_loss:.4f} val_acc={val_acc:.4f} val_bal={val_bal:.4f} '
                  f'best={best_score:.4f} lr={optimizer.param_groups[0]["lr"]:.2e}')

            if no_improve >= patience_early:
                print(f'Early stopping at epoch {epoch+1}.')
                break

    # Restore best weights (best_state always has bare keys)
    if best_state is not None:
        if is_dp:
            model.module.load_state_dict(best_state)
        else:
            model.load_state_dict(best_state)

    # Training curves (the *_curves.png paths passed by Cells 12-13 were never written before)
    if save_path and history['val_epoch']:
        try:
            fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
            ax[0].plot(range(1, len(history['train_loss']) + 1), history['train_loss'])
            ax[0].set(title='train loss', xlabel='epoch')
            ax[1].plot(history['val_epoch'], history['val_acc'], label='accuracy')
            ax[1].plot(history['val_epoch'], history['val_bal_acc'], label='balanced accuracy')
            if history['best_epoch']:
                ax[1].axvline(history['best_epoch'], ls=':', c='k', label=f'best (epoch {history["best_epoch"]})')
            ax[1].set(title=f'validation (selection: {SELECT_METRIC})', xlabel='epoch'); ax[1].legend(fontsize=8)
            ax[2].semilogy(history['val_epoch'], history['lr']); ax[2].set(title='learning rate', xlabel='epoch')
            fig.tight_layout(); fig.savefig(save_path, dpi=100); plt.close(fig)
            print(f'  training curves -> {save_path}')
        except Exception as e:   # never fail a training run over a plot
            print(f'  (could not write training curves to {save_path}: {e!r})')

    return model, history, best_score

print('Training infrastructure defined')

In [ ]:
# Cell 10: Load Pre-trained Models from the uploaded Kaggle dataset
#
# Expects the official BatSpot models (extracted from BatSpot_article):
#   <dataset>/batspot/models_call_detector/m03|m09|m11/train/ANIMAL-SPOT.pk
#   <dataset>/batspot/models_call_classifier/m09/train/ANIMAL-SPOT.pk
#
# NOTE: all four files share the basename ANIMAL-SPOT.pk, so each is staged
# here under a UNIQUE name. Copying by basename would let the classifier
# overwrite the detector and silently fine-tune the detector from the
# 15-class classifier weights.

os.makedirs(PRETRAINED_DIR, exist_ok=True)

# --- Priority 1: explicit paths from Cell 2 ---
_official = {}
_manual_set = set()
_missing_manual = []
_manual_set_attempted = False
if USE_MANUAL_MODEL_PATHS:
    for _lbl, _role, _mic, _p in [
            ('detector m03', 'detector', 'm03', DETECTOR_M03_PATH),
            ('detector m09', 'detector', 'm09', DETECTOR_M09_PATH),
            ('detector m11', 'detector', 'm11', DETECTOR_M11_PATH),
            ('classifier m09', 'classifier', 'm09', CLASSIFIER_M09_PATH)]:
        if _p is None:
            continue
        _manual_set_attempted = True
        if os.path.isfile(_p):
            _ap = os.path.abspath(_p)
            _manual_set.add(_ap)
            _official.setdefault(_role, {})[_mic] = _ap
        else:
            _missing_manual.append(f'{_lbl}: {_p}')
    if _official:
        print(f'Using {sum(len(v) for v in _official.values())} manual path(s) from Cell 2.')
    for _m in _missing_manual:
        print(f'  manual path NOT FOUND -> {_m}')
    if _missing_manual:
        print('  falling back to auto-discovery for the missing ones.')
else:
    print('USE_MANUAL_MODEL_PATHS = False -> ignoring Cell 2 paths.')

# --- Priority 2: discover the dataset directory (local repo or /kaggle/input/*) ---
BATSPOT_ARTICLE_DIR = None
_MODEL_GLOB = os.path.join('models_*', '*', 'train', 'ANIMAL-SPOT.pk')

for _cand in ['BatSpot_article/batspot', 'BatSpot_article']:
    if glob.glob(os.path.join(_cand, _MODEL_GLOB)):
        BATSPOT_ARTICLE_DIR = _cand
        break

if BATSPOT_ARTICLE_DIR is None:
    for _root in sorted(glob.glob('/kaggle/input/*')):
        for _cand in [os.path.join(_root, 'batspot'), _root]:
            if glob.glob(os.path.join(_cand, _MODEL_GLOB)):
                BATSPOT_ARTICLE_DIR = _cand
                break
        if BATSPOT_ARTICLE_DIR:
            break

# --- Collect sources, keyed role -> mic -> path ---
#
# _official ALREADY holds whatever the manual block above resolved. Do NOT
# re-initialise it here: doing so discards the Cell 2 paths and makes this
# cell report "No pre-trained models found" even though the manual lookup
# just succeeded. Discovery only fills roles/mics the manual block missed.
_manual_n = sum(len(v) for v in _official.values())
_skipped = []
if BATSPOT_ARTICLE_DIR:
    for _p in sorted(glob.glob(os.path.join(BATSPOT_ARTICLE_DIR, _MODEL_GLOB))):
        _parts = os.path.relpath(_p, BATSPOT_ARTICLE_DIR).split(os.sep)
        # e.g. ['models_call_detector', 'm09', 'train', 'ANIMAL-SPOT.pk']
        if len(_parts) < 4:
            continue
        _family, _mic = _parts[0], _parts[1]
        # Resolve the role FIRST, and only ever to 'classifier' or 'detector'.
        # buzz_detector / social_detector use different band limits and are
        # not fine-tuned here; classifying them as detectors would feed their
        # weights into the call-detector slot.
        if _family == 'models_call_classifier':
            _role = 'classifier'
        elif _family == 'models_call_detector':
            _role = 'detector'
        else:
            _skipped.append(f'{_family}/{_mic}')
            continue
        if _mic in _official.get(_role, {}):
            continue  # already supplied manually via Cell 2 (manual wins)
        _official.setdefault(_role, {})[_mic] = _p
    if _manual_n:
        print(f'Auto-discovery added nothing (Cell 2 supplied {_manual_n} model(s)); '
              f'manual paths stand.')

# --- Stage under unique names and read REAL metadata from each .pk ---
# PRETRAINED_MODELS[role][mic] = {path, sr, classes, num_classes, data, source}
PRETRAINED_MODELS = {}
detector_path = None
classifier_path = None

if _official:
    if BATSPOT_ARTICLE_DIR:
        print(f'BatSpot_article dataset: {BATSPOT_ARTICLE_DIR}\n')
    for _role, _mics in _official.items():
        for _mic, _src in sorted(_mics.items()):
            _dest = os.path.join(PRETRAINED_DIR, f'official_{_role}_{_mic}.pk')
            shutil.copy2(_src, _dest)
            try:
                _o = torch.load(_dest, map_location='cpu', weights_only=False)
                _d = _o.get('dataOpts', {})
                PRETRAINED_MODELS.setdefault(_role, {})[_mic] = {
                    'path': _dest,
                    'data': _d,
                    'sr': _d.get('sr'),
                    'classes': _o.get('classes', {}),
                    'num_classes': _o.get('classifierOpts', {}).get('num_classes'),
                    'source': _src,
                    'origin': 'manual' if os.path.abspath(_src) in _manual_set else 'discovered',
                }
            except Exception as e:
                print(f'  WARNING: could not read {_dest}: {e}')
    print('Official models staged:')
    for _role in sorted(PRETRAINED_MODELS):
        for _mic, _info in sorted(PRETRAINED_MODELS[_role].items()):
            _sr = _info['sr'] or 0
            print(f'  {_role:9s} {_mic}: {_info["num_classes"]:2d} classes, {_sr//1000:3d} kHz'
                  f'  [{_info.get("origin", "?")}] <- {_info["source"]}')
    # m09 is the paper's primary microphone (call detector test F1 0.985).
    # Prefer it; fall back to the lowest-numbered variant.
    for _role in ('detector', 'classifier'):
        _mics = PRETRAINED_MODELS.get(_role, {})
        if not _mics:
            continue
        _pick = 'm09' if 'm09' in _mics else sorted(_mics)[0]
        if _role == 'detector':
            detector_path = _mics[_pick]['path']
        else:
            classifier_path = _mics[_pick]['path']
        print(f'  -> fine-tuning {_role} from {_pick}')
    if _skipped:
        print(f'  (skipped, not fine-tuned here: {sorted(set(_skipped))})')
elif not _official:
    print('No pre-trained models found.')
    if USE_MANUAL_MODEL_PATHS and _manual_set_attempted:
        print('All Cell 2 paths were set but none resolved to a readable file.')
        print('Re-run Cell 2 and check the per-path OK / NOT FOUND lines above it.')
    else:
        print('Either set DETECTOR_M03_PATH / DETECTOR_M09_PATH / DETECTOR_M11_PATH /')
        print('CLASSIFIER_M09_PATH in Cell 2, or attach the dataset via Add Data.')

# --- Verify our spectrogram config matches each official model's dataOpts ---
print('\nConfig vs model dataOpts check:')
for _role, _cfg in (('detector', DET_CONFIG), ('classifier', CLS_CONFIG)):
    _mics = PRETRAINED_MODELS.get(_role)
    if not _mics:
        continue
    for _mic, _info in sorted(_mics.items()):
        _d = _info['data']
        _bad = []
        for _k in ('sr', 'n_fft', 'hop_length', 'fmin', 'fmax', 'n_freq_bins', 'freq_compression'):
            if _k in _d and _k in _cfg and _d[_k] != _cfg[_k]:
                _bad.append(f'{_k}: model={_d[_k]} config={_cfg[_k]}')
        _tag = f'official {_role} ({_mic})'
        if _bad:
            print(f'  MISMATCH {_tag}: ' + '; '.join(_bad))
        else:
            print(f'  OK        {_tag}')

# --- Class-overlap report (official models use European species codes) ---
if 'classes' in dir() or 'classes' in globals():
    _mine = list(globals().get('classes', []))
    for _role, _cfg in (('detector', DET_CONFIG), ('classifier', CLS_CONFIG)):
        for _mic, _info in sorted(PRETRAINED_MODELS.get(_role, {}).items()):
            _theirs = set(_info['classes'].keys())
            _ov = sorted(set(_mine) & _theirs)
            print(f'\n{_role} ({_mic}) knows {len(_theirs)} classes; '
                  f'overlap with your data {sorted(_mine)}: {_ov or "NONE"}')
            if not _ov and _role == 'classifier':
                print('  -> classifier head cannot transfer; only the encoder will be used.')
            elif _ov and _role == 'detector':
                print('  -> evaluated on the overlapping classes only (noise).')

# --- Fallback: local Data/model_output (384 kHz, 3-class) ---
if detector_path is None and classifier_path is None:
    _local = None
    for _cand in ['Data/model_output/ANIMAL-SPOT.pk',
                  '/kaggle/input/batspot-data/model_output/ANIMAL-SPOT.pk']:
        if os.path.isfile(_cand):
            _local = _cand
            break
    if _local is None and MODEL_OUTPUT_DIR and os.path.isdir(MODEL_OUTPUT_DIR):
        for _root, _, _files in os.walk(MODEL_OUTPUT_DIR):
            for _f in _files:
                if _f.endswith('.pk'):
                    _local = os.path.join(_root, _f)
                    break
            if _local:
                break
    if _local:
        _dest = os.path.join(PRETRAINED_DIR, 'fallback_local.pk')
        shutil.copy2(_local, _dest)
        detector_path = _dest
        classifier_path = _dest
        print(f'\nFallback local model: {_local}')
        print('  NOTE: 384 kHz / 3-class model. Spectrogram params will NOT match')
        print('        DET_CONFIG or CLS_CONFIG, so encoder transfer is unreliable.')
    else:
        print('\nWARNING: no pre-trained model found. Both models train from scratch.')

print(f'\nFine-tuning starting points:')
print(f'  Detector:   {detector_path}')
print(f'  Classifier: {classifier_path}')

In [ ]:
# Cell 11: Build Detector Dataset (shared across all 3 detector variants)
#
# Uses the shared train_wavs / val_wavs / test_wavs split. Every non-noise
# species is folded to label 1 (call); noise is mapped to 0.

_det_all = list(train_wavs) + list(val_wavs) + list(test_wavs)
_det_species = sorted({get_class_from_filename(f) for f in _det_all})
if 'noise' not in _det_species:
    print("WARNING: no 'noise' files in the dataset; detector sees only calls.")

# All species except 'noise' -> label 1 (call)
det_class_to_idx = {c: (0 if c == 'noise' else 1) for c in _det_species}
det_class_to_idx.setdefault('noise', 0)
det_class_to_idx.setdefault('call', 1)
det_idx_to_class = {0: 'noise', 1: 'call'}

print(f'Detector: {len(_det_species)} dataset classes -> 2 binary labels')
print(f'  label 0 (noise): {[c for c, v in det_class_to_idx.items() if v == 0]}')
print(f'  label 1 (call):  {[c for c, v in det_class_to_idx.items() if v == 1]}')

# 20 ms windows: train = random loud crop per epoch, val/test = top-k windows averaged per clip
det_seq_len = int(float(DET_CONFIG['sequence_len']) / 1000 * DET_CONFIG['sr'] / DET_CONFIG['hop_length'])
print(f'Detector: sr={DET_CONFIG["sr"]}, window={det_seq_len} frames (20 ms), WINDOW_MODE={WINDOW_MODE!r}')

det_train = WindowedBatDataset(train_wavs, det_class_to_idx, DET_CONFIG, train=True,
                               augment=USE_AUGMENTATION)
det_val   = WindowedBatDataset(val_wavs,   det_class_to_idx, DET_CONFIG, train=False)
det_test  = WindowedBatDataset(test_wavs,  det_class_to_idx, DET_CONFIG, train=False)
print(f'Detector eval windows: val={len(det_val)} ({len(det_val.file_names)} clips), '
      f'test={len(det_test)} ({len(det_test.file_names)} clips)')

# Show binary class distribution
det_train_labels = [det_class_to_idx[get_class_from_filename(f)] for f in train_wavs]
det_val_labels   = [det_class_to_idx[get_class_from_filename(f)] for f in val_wavs]
det_test_labels  = [det_class_to_idx[get_class_from_filename(f)] for f in test_wavs]
print(f'Detector train: {Counter(det_train_labels)}  (0=noise, 1=call)')
print(f'Detector val:   {Counter(det_val_labels)}')
print(f'Detector test:  {Counter(det_test_labels)}')

In [ ]:
# Cell 12: Train ALL 3 detector variants (m03, m09, m11) + evaluate each individually.
#
# PRETRAINED_MODELS['detector'] = {'m03': {...}, 'm09': {...}, 'm11': {...}}
# For each mic: load its pretrained encoder, fine-tune, evaluate on det_test.
# Results stored in det_results[mic].

def evaluate_model(model, dataset, class_names, device, model_name='Model'):
    """Evaluate on a windowed eval dataset (clip-level, windows averaged). Returns metrics dict."""
    all_probs, all_labels = predict_proba(model, dataset, device)
    all_preds = all_probs.argmax(axis=1)
    eval_labels = list(range(len(class_names)))
    acc  = accuracy_score(all_labels, all_preds)
    bal  = balanced_accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)
    report = classification_report(all_labels, all_preds, labels=eval_labels,
                                   target_names=class_names, zero_division=0,
                                   output_dict=True)
    cm = confusion_matrix(all_labels, all_preds, labels=eval_labels)
    print(f'\n=== {model_name} ===')
    print(f'Accuracy: {acc:.4f}  Balanced acc: {bal:.4f}  Precision: {prec:.4f}  '
          f'Recall: {rec:.4f}  F1: {f1:.4f}')
    print(classification_report(all_labels, all_preds, labels=eval_labels,
                                target_names=class_names, zero_division=0))
    return {'accuracy': acc, 'balanced_accuracy': bal, 'precision': prec, 'recall': rec, 'f1': f1,
            'predictions': all_preds, 'labels': all_labels, 'probs': all_probs,
            'confusion_matrix': cm, 'report': report, 'class_names': class_names}


encoderOpts_det = {'input_channels': 1, 'conv_kernel_size': 7, 'max_pool': 2, 'resnet_size': 18}
_det_mics = sorted(PRETRAINED_MODELS.get('detector', {}).keys())
if not _det_mics:
    print('No pretrained detector variants found -> training one from scratch.')
    _det_mics = ['scratch']

# det_results[mic] = {'model', 'metrics', 'best_val_acc', 'encoderOpts', 'classifierOpts'}
det_results = {}

for _mic in _det_mics:
    print(f'\n{"="*60}')
    print(f'DETECTOR VARIANT: {_mic}')
    print(f'{"="*60}')

    set_seed(SEED)   # same init + sampling order for every variant (Cell 2)
    model, encoder, _cls_head, classifierOpts = build_model(
        encoderOpts_det, DET_CONFIG['num_classes'], DEVICE)

    _pk = PRETRAINED_MODELS.get('detector', {}).get(_mic, {}).get('path')
    if _pk:
        try:
            _pre, _, _pre_opts = load_model_from_pk(_pk, DEVICE)
            encoder.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
            print(f'Encoder loaded from {os.path.basename(_pk)} ({_pre_opts.get("sr", 0)//1000}kHz)')
        except Exception as _e:
            print(f'Could not load encoder for {_mic}: {_e}')
    else:
        print(f'No pretrained path for {_mic} -> random init.')

    model = model.to(DEVICE)
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

    model, _hist, best_acc = train_model(
        model, det_train, det_val, DET_CONFIG, DEVICE,
        save_path=os.path.join(WORKING_DIR, f'detector_{_mic}_curves.png'),
    )
    print(f'Best val {SELECT_METRIC} ({_mic}): {best_acc:.4f}')

    metrics = evaluate_model(model, det_test, DET_CLASSES, DEVICE,
                             f'Detector {_mic} (Test Set)')

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(metrics['confusion_matrix'], interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=range(2), yticks=range(2),
           xticklabels=DET_CLASSES, yticklabels=DET_CLASSES,
           xlabel='Predicted', ylabel='True',
           title=f'Detector {_mic} Confusion Matrix (Test)')
    for i in range(2):
        for j in range(2):
            v = metrics['confusion_matrix'][i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > metrics['confusion_matrix'].max()/2 else 'black')
    plt.tight_layout(); plt.show()

    det_results[_mic] = {'model': model, 'metrics': metrics,
                         'best_val_acc': best_acc, 'history': _hist,
                         'encoderOpts': encoderOpts_det,
                         'classifierOpts': classifierOpts}

# Comparison table
print(f'\n{"="*60}')
print('DETECTOR COMPARISON SUMMARY')
print(f'{"="*60}')
print(f'{"Variant":<10} {"Val score":>9} {"Test Acc":>10} {"Bal Acc":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-'*70)
for _mic, _r in det_results.items():
    m = _r['metrics']
    print(f'{_mic:<10} {_r["best_val_acc"]:>9.4f} {m["accuracy"]:>10.4f} {m["balanced_accuracy"]:>8.4f} '
          f'{m["precision"]:>8.4f} {m["recall"]:>8.4f} {m["f1"]:>8.4f}')

In [ ]:
# Cell 13: Build Classifier Dataset + Train Classifier (single, shared across all pipelines)

cls_seq_len = int(float(CLS_CONFIG['sequence_len']) / 1000 * CLS_CONFIG['sr'] / CLS_CONFIG['hop_length'])
print(f'Classifier: sr={CLS_CONFIG["sr"]}, window={cls_seq_len} frames (20 ms), WINDOW_MODE={WINDOW_MODE!r}')

cls_train = WindowedBatDataset(train_wavs, CLS_CLASS_TO_IDX, CLS_CONFIG, train=True,
                               augment=USE_AUGMENTATION)
cls_val   = WindowedBatDataset(val_wavs,   CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)
cls_test  = WindowedBatDataset(test_wavs,  CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)

print(f'Classifier train: {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in train_wavs])}')
print(f'Classifier val:   {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in val_wavs])}')
print(f'Classifier test:  {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in test_wavs])}')

encoderOpts_cls = {'input_channels': 1, 'conv_kernel_size': 7, 'max_pool': 2, 'resnet_size': 18}
set_seed(SEED)
cls_model, cls_encoder, cls_classifier_head, cls_classifierOpts = build_model(
    encoderOpts_cls, CLS_CONFIG['num_classes'], DEVICE)

if classifier_path:
    try:
        _pre, _, _pre_opts = load_model_from_pk(classifier_path, DEVICE)
        cls_encoder.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
        print(f'Encoder loaded from {os.path.basename(classifier_path)} ({_pre_opts.get("sr",0)//1000}kHz)')
    except Exception as _e:
        print(f'Could not load classifier pretrained: {_e}')
else:
    print('No pre-trained classifier. Training from scratch.')

cls_model = cls_model.to(DEVICE)
print(f'Classifier params: {sum(p.numel() for p in cls_model.parameters()):,}')

cls_model, cls_history, cls_best_acc = train_model(
    cls_model, cls_train, cls_val, CLS_CONFIG, DEVICE,
    save_path=os.path.join(WORKING_DIR, 'classifier_curves.png'),
)
print(f'\nClassifier training complete. Best val {SELECT_METRIC}: {cls_best_acc:.4f}')

cls_metrics = evaluate_model(cls_model, cls_test, CLS_CLASSES, DEVICE, 'Classifier (Test Set)')

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cls_metrics['confusion_matrix'], interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
n_cls = len(CLS_CLASSES)
ax.set(xticks=range(n_cls), yticks=range(n_cls),
       xticklabels=CLS_CLASSES, yticklabels=CLS_CLASSES,
       xlabel='Predicted', ylabel='True', title='Classifier Confusion Matrix (Test)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for i in range(n_cls):
    for j in range(n_cls):
        v = cls_metrics['confusion_matrix'][i, j]
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cls_metrics['confusion_matrix'].max()/2 else 'black')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 14: Combined Pipeline — each detector variant (m03/m09/m11) x the single classifier.
#
# All probabilities are clip-level (softmax averaged over the clip's top-k windows).
# Three ways of combining detector + classifier are reported side by side, because on this
# dataset the classifier already has a 'noise' class and a hard detector gate can only ADD errors:
#   classifier alone : argmax of the 8-class classifier (no detector)
#   hard gate        : detector says noise (P(call) < tuned threshold) -> 'noise', else classifier
#   soft combine     : average the detector's and classifier's noise evidence, then argmax
# The detector threshold is tuned on the VALIDATION set (maximise call-class F1).

def run_combined_pipeline(det_model, mic_label, cls_model, det_val, det_test, cls_test, device):
    """Tune threshold on val, then evaluate classifier-alone / hard gate / soft combine on test."""
    assert [os.path.basename(f) for f in det_test.file_names] == \
           [os.path.basename(f) for f in cls_test.file_names], 'detector/classifier test order differs'
    noise_idx = CLS_CLASS_TO_IDX['noise']

    val_probs, val_true = predict_proba(det_model, det_val, device)
    val_call = val_probs[:, 1]
    best_t, best_f1 = 0.05, -1.0  # first grid value; m11 sat on the old 0.10 grid edge
    for t in np.arange(0.05, 0.96, 0.05):
        _, _, f1, _ = precision_recall_fscore_support(
            val_true, (val_call >= t).astype(int), average='binary', pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    print(f'  [{mic_label}] threshold={best_t:.2f}  val-F1={best_f1:.4f}')

    det_p, _ = predict_proba(det_model, det_test, device)
    p_call = det_p[:, 1]
    cls_p, clabels = predict_proba(cls_model, cls_test, device)

    alone = cls_p.argmax(axis=1)
    gate = np.where(p_call < best_t, noise_idx, alone)
    p_noise = 0.5 * cls_p[:, noise_idx] + 0.5 * (1 - p_call)
    rest = cls_p.copy(); rest[:, noise_idx] = 0
    rest = rest / np.maximum(rest.sum(axis=1, keepdims=True), 1e-9) * (1 - p_noise)[:, None]
    rest[:, noise_idx] = p_noise
    soft = rest.argmax(axis=1)

    out = {'threshold': best_t, 'variants': {}}
    for name, preds in (('classifier alone', alone), ('hard gate', gate), ('soft combine', soft)):
        prec, rec, f1, _ = precision_recall_fscore_support(
            clabels, preds, average='macro', zero_division=0)
        out['variants'][name] = {'acc': accuracy_score(clabels, preds), 'prec': prec, 'rec': rec,
                                 'f1': f1, 'cm': confusion_matrix(clabels, preds,
                                                                  labels=list(range(len(CLS_CLASSES)))),
                                 'report': classification_report(clabels, preds,
                                                                 labels=list(range(len(CLS_CLASSES))),
                                                                 target_names=CLS_CLASSES, zero_division=0)}
    # keep the original keys (hard gate) so older downstream code keeps working
    g = out['variants']['hard gate']
    out.update({'acc': g['acc'], 'prec': g['prec'], 'rec': g['rec'], 'f1': g['f1'],
                'cm': g['cm'], 'report': g['report']})
    return out


pipeline_results = {}
for _mic, _dr in det_results.items():
    print(f'\n{"="*60}')
    print(f'COMBINED PIPELINE: Detector {_mic} -> Classifier')
    print(f'{"="*60}')
    _res = run_combined_pipeline(_dr['model'], _mic, cls_model, det_val, det_test, cls_test, DEVICE)
    pipeline_results[_mic] = _res

    print(f'\n=== Combined {_mic} (hard gate): Accuracy={_res["acc"]:.4f}  F1={_res["f1"]:.4f} ===')
    print(_res['report'])

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(_res['cm'], interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    n_cls = len(CLS_CLASSES)
    ax.set(xticks=range(n_cls), yticks=range(n_cls),
           xticklabels=CLS_CLASSES, yticklabels=CLS_CLASSES,
           xlabel='Predicted', ylabel='True',
           title=f'Combined {_mic} (hard gate) Confusion Matrix (Test)')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(n_cls):
        for j in range(n_cls):
            v = _res['cm'][i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > _res['cm'].max()/2 else 'black')
    plt.tight_layout(); plt.show()

# Final comparison table
print(f'\n{"="*72}')
print('COMBINED PIPELINE COMPARISON SUMMARY (test accuracy, 8 classes)')
print(f'{"="*72}')
print(f'{"Detector":<10} {"Thresh":>7} {"Clf alone":>10} {"Hard gate":>10} {"Soft comb.":>11} {"Gate F1":>8}')
print('-'*72)
for _mic, _r in pipeline_results.items():
    v = _r['variants']
    print(f'{_mic:<10} {_r["threshold"]:>7.2f} {v["classifier alone"]["acc"]:>10.4f} '
          f'{v["hard gate"]["acc"]:>10.4f} {v["soft combine"]["acc"]:>11.4f} {_r["f1"]:>8.4f}')

In [ ]:
# Cell 15: Export Models as .pk
#
# Exports all fine-tuned detector variants + the single classifier.
import copy

def export_pk(model, encoderOpts, classifierOpts, dataOpts, classes, path):
    """Export model as .pk file compatible with BatSpot GUI/CLI."""
    if isinstance(model, nn.DataParallel):
        model = model.module
    # Export a CPU COPY. model.cpu() moved the live models off the GPU in place, so any later
    # cell that ran them on DEVICE (e.g. the inference cells) failed with a device mismatch.
    model = copy.deepcopy(model).cpu()
    encoder = model[0]
    classifier = model[1]
    torch.save({
        'encoderOpts':     encoderOpts,
        'classifierOpts':  classifierOpts,
        'dataOpts':        dataOpts,
        'encoderState':    encoder.state_dict(),
        'classifierState': classifier.state_dict(),
        'classes':         classes,
    }, path)
    print(f'Saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')

det_dataOpts = {
    'sr': DET_CONFIG['sr'], 'n_fft': DET_CONFIG['n_fft'],
    'hop_length': DET_CONFIG['hop_length'], 'n_freq_bins': DET_CONFIG['n_freq_bins'],
    'fmin': DET_CONFIG['fmin'], 'fmax': DET_CONFIG['fmax'],
    'freq_compression': DET_CONFIG['freq_compression'],
    'min_level_db': -100, 'ref_level_db': 20, 'preemphases': 0.98,
}
det_classes = {'noise': 0, 'call': 1}

# Export each detector variant
det_out_paths = {}
for _mic, _dr in det_results.items():
    _out = os.path.join(WORKING_DIR, f'detector_192khz_{_mic}.pk')
    export_pk(_dr['model'], _dr['encoderOpts'], _dr['classifierOpts'],
              det_dataOpts, det_classes, _out)
    det_out_paths[_mic] = _out

# Export classifier (250kHz, multi-species)
cls_dataOpts = {
    'sr': CLS_CONFIG['sr'], 'n_fft': CLS_CONFIG['n_fft'],
    'hop_length': CLS_CONFIG['hop_length'], 'n_freq_bins': CLS_CONFIG['n_freq_bins'],
    'fmin': CLS_CONFIG['fmin'], 'fmax': CLS_CONFIG['fmax'],
    'freq_compression': CLS_CONFIG['freq_compression'],
    'min_level_db': -100, 'ref_level_db': 20, 'preemphases': 0.98,
}
cls_out_path = os.path.join(WORKING_DIR, 'classifier_250khz.pk')
export_pk(cls_model, encoderOpts_cls, cls_classifierOpts, cls_dataOpts, CLS_CLASS_TO_IDX, cls_out_path)

# Verify exports
print('\nVerifying exports...')
for _mic, p in det_out_paths.items():
    obj = torch.load(p, map_location='cpu', weights_only=False)
    print(f'  detector_{_mic}: enc={len(obj["encoderState"])} params, '
          f'cls={len(obj["classifierState"])} params, classes={obj["classes"]}')
obj = torch.load(cls_out_path, map_location='cpu', weights_only=False)
print(f'  classifier:    enc={len(obj["encoderState"])} params, '
      f'cls={len(obj["classifierState"])} params, classes={list(obj["classes"].keys())}')

print('\nNOTE: these models were trained with per-window MIN-MAX normalisation on 20 ms windows,')
print('exactly like the official BatSpot models. When predicting with PREDICTION/start_prediction.py')
print('(or the GUI) enable min-max normalisation (min_max_norm=true), otherwise inputs are scaled differently.')

In [ ]:
# Cell 16: Model Inventory — official pre-trained models + your own models

if MODEL_OUTPUT_DIR is None:
    for candidate in ['Data/model_output', '/kaggle/input/model_output',
                      '/kaggle/input/batspot-data/model_output']:
        if os.path.isdir(candidate):
            MODEL_OUTPUT_DIR = candidate
            break

model_inventory = []
model_tags = {}

if MODEL_OUTPUT_DIR and os.path.isdir(MODEL_OUTPUT_DIR):
    for root, dirs, files in os.walk(MODEL_OUTPUT_DIR):
        for f in files:
            if f.endswith('.pk') or f.endswith('.checkpoint'):
                p = os.path.join(root, f)
                model_inventory.append(p)
                model_tags[p] = 'yours'

# Official models staged by Cell 10
for role, mics in PRETRAINED_MODELS.items():
    for mic, info in sorted(mics.items()):
        p = info['path']
        if p not in model_inventory:
            model_inventory.append(p)
        model_tags[p] = f'official:{role}/{mic}'

print(f'MODEL_OUTPUT_DIR: {MODEL_OUTPUT_DIR}')
print(f'Found {len(model_inventory)} models:\n')
for m in model_inventory:
    tag = model_tags.get(m, '?')
    size_mb = os.path.getsize(m) / 1e6
    fmt = '.pk' if m.endswith('.pk') else '.checkpoint'
    info = None
    for role, mics in PRETRAINED_MODELS.items():
        for mic, inf in mics.items():
            if inf['path'] == m:
                info = inf
    if info:
        print(f'  [{tag:22s}] {size_mb:6.1f} MB {fmt}  '
              f'{info["num_classes"]} classes, {(info["sr"] or 0)//1000} kHz')
    else:
        print(f'  [{tag:22s}] {size_mb:6.1f} MB {fmt}  {os.path.basename(m)}')


In [ ]:
# Cell 17: Load & Evaluate Each Existing Model on the TEST split
#
# WHY THIS CELL CHANGED: the previous version only scored files whose species name appeared
# in the model's own class list. The official detectors know {'noise','target'} and the official
# classifier 15 European species, so only the 186 'noise' files matched -- the reported
# "accuracy" (e.g. m03 = 0.984) was NOISE RECALL on all clips, i.e. "it says noise to almost
# everything". That number is not comparable to a fine-tuned model's accuracy on calls + noise.
#
# Now every model is scored on the SAME held-out test clips with the SAME windowed protocol:
#   * detector {noise,target}         : every bat species counts as 'target' -> call-vs-noise
#   * classifier w/o species overlap  : only "does it say noise?" is meaningful -> scored as a
#                                       noise-vs-call detector (species accuracy is N/A)
#   * model whose class names overlap : exact-name multi-class on the overlapping classes
from sklearn.metrics import roc_auc_score


def run_model_evaluation(model_path, wav_files, device):
    """Evaluate one existing model on `wav_files` (pass test_wavs)."""
    model_name = os.path.basename(model_path)
    try:
        model, model_classes, dataOpts = load_any_model(model_path, device)
    except Exception as e:
        return {'name': model_name, 'path': model_path, 'error': str(e)}
    model.to(device).eval()

    data_species = {get_class_from_filename(f) for f in wav_files} - {'noise'}
    cfg = {'sr': dataOpts.get('sr', 384000), 'n_fft': dataOpts.get('n_fft', 256),
           'hop_length': dataOpts.get('hop_length', 128),
           'n_freq_bins': dataOpts.get('n_freq_bins', 256),
           'fmin': dataOpts.get('fmin', 18000), 'fmax': dataOpts.get('fmax', 90000),
           'sequence_len': 20}

    if set(model_classes) == {'noise', 'target'}:
        kind = 'detector (call vs noise)'
        files = list(wav_files)
        label_map = {c: 0 for c in ['noise']}
        label_map.update({c: 1 for c in data_species})
        names = ['noise', 'call']
    elif 'noise' in model_classes and not (set(model_classes) & data_species):
        kind = 'classifier scored as noise-vs-call (no species overlap)'
        files = list(wav_files)
        label_map = {c: 0 for c in ['noise']}
        label_map.update({c: 1 for c in data_species})
        names = ['noise', 'call']
    else:
        kind = 'multi-class (overlapping classes only)'
        files = [f for f in wav_files if get_class_from_filename(f) in model_classes]
        label_map = {get_class_from_filename(f): model_classes[get_class_from_filename(f)] for f in files}
        names = [k for k, _ in sorted(model_classes.items(), key=lambda kv: kv[1])]
    if not files:
        return {'name': model_name, 'path': model_path, 'unsupported': True,
                'error': f'no overlapping files: model knows {sorted(model_classes)}, '
                         f'data has {sorted(data_species | {"noise"})}'}

    ds = WindowedBatDataset(files, label_map, cfg, train=False)
    probs, labels = predict_proba(model, ds, device)
    out = {'name': model_name, 'path': model_path, 'kind': kind, 'n_files': len(files),
           'target_names': names, 'labels': labels, 'files': files, 'auc': None}
    if kind.startswith('detector'):
        p_call = probs[:, model_classes['target']]
    elif kind.startswith('classifier'):
        p_call = 1.0 - probs[:, model_classes['noise']]
    else:
        p_call = None
    if p_call is not None:
        preds = (p_call >= 0.5).astype(int)
        out['auc'] = float(roc_auc_score(labels, p_call)) if len(set(labels)) > 1 else None
    else:
        preds = probs.argmax(axis=1)
    out.update({'preds': preds, 'accuracy': accuracy_score(labels, preds),
                'balanced_accuracy': balanced_accuracy_score(labels, preds),
                'confusion_matrix': confusion_matrix(labels, preds, labels=list(range(len(names)))),
                'model_classes': model_classes})
    return out


# Evaluate all models on the held-out TEST split
val_results = []
for m_path in model_inventory:
    result = run_model_evaluation(m_path, test_wavs, DEVICE)
    val_results.append(result)
    tag = model_tags.get(m_path, '?')
    if 'error' in result:
        status = 'UNSUPPORTED' if result.get('unsupported') else 'ERROR'
        print(f'[{status}] {tag} / {result["name"]}: {result["error"]}')
    else:
        _auc = f', auc={result["auc"]:.4f}' if result['auc'] is not None else ''
        print(f'[OK] {tag} / {result["name"]}: {result["kind"]}: acc={result["accuracy"]:.4f}, '
              f'bal_acc={result["balanced_accuracy"]:.4f}{_auc}, files={result["n_files"]}')

In [ ]:
# Cell 18: Results Summary & Confusion Matrices

print('\n' + '='*80)
print('EXISTING MODEL EVALUATION SUMMARY (held-out test split)')
print('='*80)

for r in val_results:
    if 'error' in r:
        print(f'\n{r["name"]}: ERROR - {r["error"]}')
        continue
    print(f'\n--- {r["name"]} ---')
    print(f'  Mode: {r["kind"]}')
    print(f'  Files evaluated: {r["n_files"]}')
    print(f'  Accuracy: {r["accuracy"]:.4f}   Balanced accuracy: {r["balanced_accuracy"]:.4f}'
          + (f'   AUC: {r["auc"]:.4f}' if r['auc'] is not None else ''))
    print(classification_report(r['labels'], r['preds'], labels=list(range(len(r['target_names']))),
                                target_names=r['target_names'], zero_division=0))

    cm = r['confusion_matrix']
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    names = r['target_names']
    n = len(names)
    ax.set(xticks=range(n), yticks=range(n), xticklabels=names, yticklabels=names,
           xlabel='Predicted', ylabel='True', title=f'{r["name"][:40]}')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.show()

# ---- Official (zero-shot) vs fine-tuned, same test clips, call-vs-noise -------------------
print('\n' + '='*80)
print('OFFICIAL (zero-shot) vs FINE-TUNED -- call vs noise on the same test clips')
print('='*80)
print(f'{"Model":<34} {"n":>4} {"Acc":>7} {"Bal acc":>8} {"AUC":>7}  Note')
print('-'*92)
for r in val_results:
    if 'error' in r or r['auc'] is None:
        continue
    _note = 'NO SIGNAL (AUC~0.5): plain accuracy just reflects the class mix' if abs(r['auc'] - 0.5) < 0.1 \
        else r['kind']
    print(f'{("official " + r["name"].replace(".pk", ""))[:34]:<34} {r["n_files"]:>4} {r["accuracy"]:>7.4f} '
          f'{r["balanced_accuracy"]:>8.4f} {r["auc"]:>7.4f}  {_note}')
for _mic, _dr in det_results.items():
    _m = _dr['metrics']
    _auc = roc_auc_score(_m['labels'], _m['probs'][:, 1]) if len(set(_m['labels'])) > 1 else float('nan')
    print(f'{"fine-tuned detector " + _mic:<34} {len(_m["labels"]):>4} {_m["accuracy"]:>7.4f} '
          f'{_m["balanced_accuracy"]:>8.4f} {_auc:>7.4f}  fine-tuned call-vs-noise')

# Balanced accuracy is the fair comparison: on an 81/19 call/noise split plain accuracy rewards
# a model that answers "call" for everything.
print('\nBalanced accuracy, official zero-shot -> fine-tuned (same test clips):')
for _mic, _dr in det_results.items():
    _off = [r for r in val_results if 'error' not in r and r['name'] == f'official_detector_{_mic}.pk']
    if _off:
        _o, _f = _off[0]['balanced_accuracy'], _dr['metrics']['balanced_accuracy']
        print(f'  {_mic}: {_o:.3f} -> {_f:.3f}  ({_f - _o:+.3f})')

In [ ]:
# Cell 19: Example Predictions from Each Model (random test clips)

for r in val_results:
    if 'error' in r:
        continue
    names = r['target_names']
    print(f'\n--- {r["name"]} ({r["kind"]}) ---')
    print(f'Classes: {names}')

    n_show = min(8, len(r['preds']))
    indices = np.random.choice(len(r['preds']), n_show, replace=False)
    for idx in sorted(indices):
        true_cls = names[r['labels'][idx]] if r['labels'][idx] < len(names) else '?'
        pred_cls = names[r['preds'][idx]] if r['preds'][idx] < len(names) else '?'
        fname = os.path.basename(r['files'][idx])
        match = 'OK' if r['labels'][idx] == r['preds'][idx] else 'MISS'
        print(f'  {match} true={true_cls:6s} pred={pred_cls:6s}  {fname}')

## Inference — run the best detector + classifier on new recordings (Cells 21–24)

Upload the test recordings as a Kaggle dataset (**Add Data → Upload**, the folder or a `.zip`), paste
its path into `INFER_INPUT_DIR` in Cell 21, and run Cells 21–23. Any layout works; e.g. 7 folders × 12
`.wav` files, each holding one or many calls.

| Cell | What it does |
| :--- | :--- |
| 21 | Settings + picks the combo: the detector variant with the best **validation** score (not test) and the fine-tuned classifier. `INFER_DETECTOR_PK` / `INFER_CLASSIFIER_PK` load exported `.pk` files instead, so you can skip training (run Cells 2–6 first). |
| 22 | Functions: sliding-window detector scan (official BatSpot: 20 ms windows, 10 ms hop, threshold 0.5), merging into selections, classification of each selection with the test-clip protocol, Low/High/Peak frequency, clock time. |
| 23 | Runs every recording and writes **`/kaggle/working/batspot_detections.txt`** (below), selections the classifier calls `noise` to `batspot_detections_rejected_noise.txt`, and one Raven table per recording to `raven_tables/`. |
| 24 | Optional: if you set `INFER_TRUTH_DIR` to your own Raven selection tables, scores the output (boxes found, species accuracy, false alarms on noise boxes). |

Output format (comma-separated, one row per selection, `Selection` numbered over the whole file):

```
Selection,name_of_file,Channel,Begin Time (s),End Time (s),Begin Clock Time,End Clock Time,Low Freq (Hz),High Freq (Hz),Peak Freq (Hz),Delta Time (s),Species detected,Confidence
```

* **Clock times** come from the recording start in the file name (`YYYYMMDD_HHMMSS`, as Raven did for the
  training tables), else from the AudioMoth header comment, else they count from `00:00:00` (reported).
* **Confidence** is the classifier's probability for the species it reports (softmax averaged over the
  selection's top-5 windows). It is a model score, not a calibrated probability of being right.
* **The scan settings were checked on this dataset's own recordings** (`AGENTS.md` §10). If calls are
  missed, lower `INFER_DET_THRESHOLD`; if there are too many short noise selections, raise it or
  `INFER_MIN_WINDOWS`.

In [ ]:
# Cell 21: INFERENCE on new recordings -- configuration + pick the best detector/classifier combo
#
# Upload the test recordings as a Kaggle dataset (Add Data -> Upload -> the folder, or a .zip of
# it) and paste its path ("Copy path") into INFER_INPUT_DIR. Expected layout (any depth works):
#     <INFER_INPUT_DIR>/<folder 1>/*.wav  ...  <folder 7>/*.wav      e.g. 7 folders x 12 recordings
# Every .wav / .WAV below INFER_INPUT_DIR is processed; each may hold zero, one or many calls.
#
# What happens to each recording (the BatSpot workflow: the detector scans, the classifier labels):
#   1. DETECTOR   20 ms windows every INFER_HOP_S over the whole file -- official BatSpot scan
#                 settings (10 ms hop, threshold 0.5) -- with the training front end and the same
#                 per-window min-max normalisation.
#   2. SELECTIONS windows with P(call) >= INFER_DET_THRESHOLD are merged into one selection when
#                 they are at most INFER_MERGE_GAP_S apart (the pulses of one pass), mirroring the
#                 ~0.4 s bouts in the training selection tables.
#   3. CLASSIFIER each selection is scored exactly like a test clip: the TEST_TOPK loudest 20 ms
#                 windows, softmax averaged. Species detected = argmax, Confidence = that averaged
#                 probability. Selections the classifier labels 'noise' go to *_rejected_noise.txt.
#   4. MEASURE    Low / High / Peak frequency from a ~375 Hz-resolution spectrogram (Raven's
#                 resolution for these 384 kHz files); clock times from the recording start time
#                 in the file name (YYYYMMDD_HHMMSS) or the AudioMoth header.
#
# Models: by default the ones trained above -- the detector variant with the best VALIDATION score
# (never the test score) plus the classifier. To skip training: run Cells 2-6, then Cells 21-24 with
# INFER_DETECTOR_PK / INFER_CLASSIFIER_PK pointing at .pk files exported by Cell 15.

INFER_INPUT_DIR = '/kaggle/input/<your-test-dataset>'    # folder (or .zip) with the test recordings
INFER_OUTPUT_FILE = os.path.join(WORKING_DIR, 'batspot_detections.txt')
INFER_DETECTOR = 'auto'        # 'auto' = best validation score of the detectors trained above, or 'm03' / 'm09' / 'm11'
INFER_DETECTOR_PK = None       # optional: exported detector .pk, e.g. '/kaggle/input/<run>/detector_192khz_m09.pk'
INFER_CLASSIFIER_PK = None     # optional: exported classifier .pk, e.g. '/kaggle/input/<run>/classifier_250khz.pk'

INFER_DET_THRESHOLD = 0.5      # window P(call) needed to open / extend a selection (official BatSpot value)
INFER_HOP_S = 0.010            # detector window hop in seconds (official BatSpot value)
INFER_MERGE_GAP_S = 0.10       # positive windows closer than this join one selection
INFER_MIN_WINDOWS = 2          # selections supported by fewer positive windows are dropped (isolated clicks)
INFER_MAX_SELECTION_S = 1.0    # longer selections (continuous activity) are split at their widest silence
INFER_DROP_NOISE = True        # True: classifier-'noise' selections go to *_rejected_noise.txt, not the main file
INFER_FREQ_FLOOR_HZ = 10000    # ignore energy below this when measuring Low/High/Peak (AudioMoth high-pass = 10 kHz)
INFER_FREQ_CEIL_HZ = 150000    # ...and above this (99.7 % of the annotated boxes end below 150 kHz)
INFER_FREQ_SNR_DB = 12         # Low/High = contiguous band around the peak at least this far above the background
                               # (Low is unreliable for faint rhle/rhro CF calls -- see Cell 22)
INFER_CHUNK_S = 60             # files are scanned in chunks of this many seconds (bounded memory for long files)
INFER_AMP = True               # fp16 for the detector scan on GPU (as in training): ~1.7x faster, P(call) moves
                               # by <= 0.003 (4 of 30 000 windows flipped at 0.5); the classifier stays fp32
INFER_NAME_WITH_FOLDER = True  # name_of_file = '<folder>/<file>.wav' (AudioMoth file names repeat across sites)
INFER_WRITE_RAVEN_TABLES = True  # also write one tab-separated Raven selection table per recording
INFER_TRUTH_DIR = None         # optional: folder with YOUR Raven selection tables for these files -> Cell 24 scores the output


def _unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m


def _model_from_pk(path):
    """Load an exported (or official) .pk for inference. Returns (model, names, cfg)."""
    model, classes, d = load_model_from_pk(path, DEVICE)
    inv = {v: k for k, v in classes.items()}
    names = [inv.get(i, f'class{i}') for i in range(model[1].linear.out_features)]
    cfg = {'sr': d['sr'], 'n_fft': d.get('n_fft', 256), 'hop_length': d.get('hop_length', 128),
           'n_freq_bins': d.get('n_freq_bins') or d.get('num_mels') or 256,
           'fmin': d.get('fmin'), 'fmax': d.get('fmax'), 'sequence_len': 20}
    return model, names, cfg


# ---- detector -------------------------------------------------------------------------------
if INFER_DETECTOR_PK:
    _m, _names, _cfg = _model_from_pk(INFER_DETECTOR_PK)
    _call = [i for i, n in enumerate(_names) if n in ('call', 'target')]
    assert len(_names) == 2 and _call, f'{INFER_DETECTOR_PK} is not a call/noise detector: {_names}'
    INFER_DET = {'model': _m, 'cfg': _cfg, 'call_idx': _call[0],
                 'label': f'detector from {os.path.basename(INFER_DETECTOR_PK)}'}
else:
    assert globals().get('det_results'), ('No trained detector in memory: run Cells 7-13 first, '
                                          'or set INFER_DETECTOR_PK to an exported detector .pk')
    print(f'Detector variants (selection uses the VALIDATION {SELECT_METRIC}; test shown for reference):')
    for _mic, _r in det_results.items():
        print(f'  {_mic:<8} val {_r["best_val_acc"]:.4f}   test acc {_r["metrics"]["accuracy"]:.4f}  '
              f'test bal acc {_r["metrics"]["balanced_accuracy"]:.4f}')
    if INFER_DETECTOR == 'auto':
        # ties -> m09, the paper's primary microphone
        _mic = max(det_results, key=lambda m: (round(det_results[m]['best_val_acc'], 6), m == 'm09'))
    else:
        assert INFER_DETECTOR in det_results, f'{INFER_DETECTOR!r} not in {list(det_results)}'
        _mic = INFER_DETECTOR
    INFER_DET = {'model': _unwrap(det_results[_mic]['model']), 'cfg': DET_CONFIG, 'call_idx': 1,
                 'label': f'fine-tuned detector {_mic} (val {SELECT_METRIC} '
                          f'{det_results[_mic]["best_val_acc"]:.4f})'}

# ---- classifier -----------------------------------------------------------------------------
if INFER_CLASSIFIER_PK:
    _m, _names, _cfg = _model_from_pk(INFER_CLASSIFIER_PK)
    INFER_CLS = {'model': _m, 'cfg': _cfg, 'names': _names,
                 'label': f'classifier from {os.path.basename(INFER_CLASSIFIER_PK)}'}
else:
    assert 'cls_model' in globals(), ('No trained classifier in memory: run Cell 13 first, '
                                      'or set INFER_CLASSIFIER_PK to an exported classifier .pk')
    INFER_CLS = {'model': _unwrap(cls_model), 'cfg': CLS_CONFIG, 'names': list(CLS_CLASSES),
                 'label': f'fine-tuned classifier (val {SELECT_METRIC} {cls_best_acc:.4f})'}
INFER_CLS['noise_idx'] = INFER_CLS['names'].index('noise') if 'noise' in INFER_CLS['names'] else None

for _r in (INFER_DET, INFER_CLS):
    _r['model'] = _r['model'].to(DEVICE).eval()   # Cell 15 used to leave the live models on the CPU

print(f'\nINFERENCE COMBO')
print(f'  detector  : {INFER_DET["label"]}  ({INFER_DET["cfg"]["sr"]//1000} kHz, '
      f'{INFER_DET["cfg"]["fmin"]}-{INFER_DET["cfg"]["fmax"]} Hz)')
print(f'  classifier: {INFER_CLS["label"]}  ({INFER_CLS["cfg"]["sr"]//1000} kHz, classes {INFER_CLS["names"]})')
print(f'  scan: {INFER_HOP_S*1000:.0f} ms hop, threshold {INFER_DET_THRESHOLD}, merge gap '
      f'{INFER_MERGE_GAP_S*1000:.0f} ms, min {INFER_MIN_WINDOWS} windows, drop classifier-noise: {INFER_DROP_NOISE}')
if INFER_CLS['noise_idx'] is None:
    print('  NOTE: the classifier has no "noise" class, so every detector selection gets a species.')

In [ ]:
# Cell 22: Inference functions -- detector scan, selections, classification, measurements, tables
import re as _re_inf
import csv as _csv
import time as _time
import datetime as _dt
import scipy.signal as _ss
import resampy.filters as _rf

# ---- resampling, identical to training (resampy 'kaiser_best') ------------------------------
# Resampling a 5-min 384 kHz file with resampy takes ~40 s (about an hour for 84 such files on
# Kaggle). For an exact k:1 decimation (384 -> 192 kHz for the detector) resampy's kaiser_best
# filter reduces to a fixed symmetric 199-tap FIR, applied here as a strided convolution on the GPU
# (scipy's resample_poly on CPU) with the same output (max|diff| ~1e-7, float32 rounding). A
# self-check against resampy runs once per ratio; any other ratio (e.g. 384 -> 250 kHz for the
# classifier, which only resamples the selections) uses resampy itself.
_FIR, _FIR_OK = {}, {}


def _kaiser_best_fir(k):
    if k not in _FIR:
        hw, prec, _ = _rf.get_filter('kaiser_best')
        step = prec // k
        if step * k != prec:
            _FIR[k] = None
        else:
            w = hw[np.arange(0, len(hw), step)][: len(hw) // step] / k
            _FIR[k] = np.concatenate([w[:0:-1], w])
    return _FIR[k]


def _fir_decimate(x, h, k):
    """y[t] = sum_m h[m] x[k*t + m] (zeros outside x): the FIR above, on the GPU when there is one."""
    if DEVICE.type != 'cuda':
        return _ss.resample_poly(x.astype(np.float64), 1, k, window=h.copy()).astype(np.float32)
    half = (len(h) - 1) // 2
    xt = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(DEVICE)[None, None]
    wt = torch.from_numpy(h.astype(np.float32)).to(DEVICE)[None, None]
    tf32 = torch.backends.cudnn.allow_tf32
    torch.backends.cudnn.allow_tf32 = False     # TF32 would cost ~3 digits; keep full float32
    try:
        y = torch.nn.functional.conv1d(xt, wt, stride=k, padding=half)
    finally:
        torch.backends.cudnn.allow_tf32 = tf32
    return y[0, 0].cpu().numpy()


def resample_like_training(x, sr_in, sr_out):
    x = np.asarray(x, dtype=np.float32)
    if sr_in == sr_out:
        return x
    k = sr_in // sr_out if sr_in % sr_out == 0 else None
    h = _kaiser_best_fir(k) if k else None
    if h is not None and k not in _FIR_OK:
        probe = np.random.default_rng(0).standard_normal(sr_in // 4).astype(np.float32) * 0.1
        ref = resampy.resample(probe, sr_in, sr_out, filter='kaiser_best')
        fast = _fir_decimate(probe, h, k)
        _FIR_OK[k] = len(ref) == len(fast) and float(np.abs(ref - fast).max()) < 1e-5
        print(f'[resample] {sr_in}->{sr_out} Hz: exact fast path ({DEVICE.type}) '
              f'{"verified against resampy" if _FIR_OK[k] else "MISMATCH -> using resampy"}')
    if h is not None and _FIR_OK.get(k):
        return _fir_decimate(x, h, k)
    return resampy.resample(x, sr_in, sr_out, filter='kaiser_best').astype(np.float32)


def _read_mono(path, n_in, start, stop):
    """Samples [start, stop) of a file as mono (channel mean, like training), zeros outside the file."""
    s, e = max(0, start), min(n_in, stop)
    if e > s:
        x, _ = sf.read(path, start=s, stop=e, dtype='float32', always_2d=True)
        x = x.mean(axis=1)
    else:
        x = np.zeros(0, np.float32)
    if start < s or stop > e:
        x = np.concatenate([np.zeros(s - start, np.float32), x,
                            np.zeros(max(0, stop - max(e, s)), np.float32)])
    return x.astype(np.float32)


def _resampled(path, n_in, sr_in, sr, r0, r1, margin=4096):
    """Samples [r0, r1) of the WHOLE recording resampled to `sr`, computed from a margin-padded
    excerpt. The resampling filters reach < 200 samples, so the result equals resampling the
    whole file at once -- which lets long files be scanned in chunks."""
    s = int(np.floor((r0 - margin) * sr_in / sr))
    e = int(np.ceil((r1 + margin) * sr_in / sr))
    y = resample_like_training(_read_mono(path, n_in, s, e), sr_in, sr)
    i0 = int(round(r0 - s * sr / sr_in))
    out = y[i0:i0 + (r1 - r0)]
    return out if len(out) == r1 - r0 else np.pad(out, (0, r1 - r0 - len(out)))


def _minmax_batch(w):
    """Per-window min-max to [0, 1] (same rule as minmax_normalize), w: (B, frames, bins) tensor."""
    w = w - w.amin(dim=(1, 2), keepdim=True)
    mx = w.amax(dim=(1, 2), keepdim=True)
    return torch.where(mx > 0, w / mx.clamp_min(1e-30), w)


@torch.no_grad()
def scan_detector(path, det, batch=1024):
    """P(call) for every 20 ms window (hop INFER_HOP_S) of a recording.
    Returns (window_begin_s, window_end_s, p_call) as numpy arrays."""
    cfg, model = det['cfg'], det['model']
    sr, hl, nfft = cfg['sr'], cfg['hop_length'], cfg['n_fft']
    seq = int(float(cfg['sequence_len']) / 1000 * sr / hl)
    step = max(1, int(round(INFER_HOP_S * sr / hl)))
    info = sf.info(path)
    sr_in, n_in = info.samplerate, info.frames
    n_res = int(np.ceil(n_in * sr / sr_in))
    n_frames = (n_res - nfft) // hl + 1 if n_res >= nfft else 0

    def _p(windows):                            # windows: (B, seq, bins) float tensor on DEVICE
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=INFER_AMP and DEVICE.type == 'cuda'):
            out = model(_minmax_batch(windows).unsqueeze(1))
        return torch.softmax(out.float(), dim=1)[:, det['call_idx']].cpu().numpy()

    if n_frames < seq:                          # shorter than one window: one zero-padded window
        y = _resampled(path, n_in, sr_in, sr, 0, max(n_res, nfft))
        db = clip_to_db_spectrogram(y, cfg)
        win = pad_window(minmax_normalize(db), seq)
        p = _p(torch.from_numpy(win[None]).to(DEVICE))
        return np.array([0.0]), np.array([n_in / sr_in]), p

    starts = np.arange(0, n_frames - seq + 1, step)
    per_chunk = max(1, int(INFER_CHUNK_S * sr / hl) // step)
    lead = 16                                   # left context frames: pre-emphasis edge falls outside
    probs = []
    for c in range(0, len(starts), per_chunk):
        ws = starts[c:c + per_chunk]
        f0, f1 = int(ws[0]), int(ws[-1]) + seq  # frames needed: [f0, f1)
        g0 = max(0, f0 - lead)
        db = clip_to_db_spectrogram(_resampled(path, n_in, sr_in, sr, g0 * hl, (f1 - 1) * hl + nfft), cfg)
        x = torch.from_numpy(np.ascontiguousarray(db[f0 - g0:])).to(DEVICE)
        W = x.unfold(0, seq, step).permute(0, 2, 1)          # (n_windows, seq, bins), a view
        assert W.shape[0] == len(ws), (W.shape, len(ws))
        for b in range(0, len(ws), batch):
            probs.append(_p(W[b:b + batch]))
    p = np.concatenate(probs)
    t0 = starts * hl / sr
    t1 = ((starts + seq - 1) * hl + nfft) / sr
    return t0, t1, p


def build_selections(t0, t1, p, threshold, merge_gap, min_windows, max_len=None):
    """Windows with p >= threshold -> selections. Windows at most `merge_gap` apart join one selection;
    a selection longer than `max_len` (continuous activity) is split at its widest internal silence,
    so long passes come out as bout-sized rows like the training selection tables."""
    max_len = INFER_MAX_SELECTION_S if max_len is None else max_len
    pos = np.flatnonzero(p >= threshold)
    if not len(pos):
        return []
    todo = np.split(pos, np.flatnonzero(t0[pos[1:]] - t1[pos[:-1]] > merge_gap) + 1)[::-1]
    groups = []
    while todo:                                  # explicit stack: long runs cannot hit recursion limits
        g = todo.pop()
        if len(g) < 2 or not max_len or t1[g[-1]] - t0[g[0]] <= max_len:
            groups.append(g)
            continue
        gaps = t0[g[1:]] - t1[g[:-1]]            # silence between consecutive positive windows
        ok = ((t1[g[:-1]] - t0[g[0]] >= max_len / 4) & (t1[g[-1]] - t0[g[1:]] >= max_len / 4))
        mid = np.abs(np.arange(len(gaps)) - (len(gaps) - 1) / 2) / len(gaps)
        # widest silence first, then the least confident boundary, then the most central one
        score = np.where(ok, gaps, -np.inf) - 1e-6 * (p[g[1:]] + p[g[:-1]]) - 1e-9 * mid
        if not np.isfinite(score).any():
            score = gaps - 1e-9 * mid
        k = int(np.argmax(score))
        todo += [g[k + 1:], g[:k + 1]]           # left piece is processed first -> time order kept
    return [{'begin': float(t0[g[0]]), 'end': float(t1[g[-1]]), 'n': len(g), 'p_max': float(p[g].max())}
            for g in groups if len(g) >= min_windows]


@torch.no_grad()
def classify_selections(path, sels, cls, batch=512):
    """Classifier probabilities per selection, using the test-clip protocol (top-k loudest
    windows, softmax averaged). Returns an (n_selections, n_classes) array."""
    cfg, model = cls['cfg'], cls['model']
    sr, hl, nfft = cfg['sr'], cfg['hop_length'], cfg['n_fft']
    seq = int(float(cfg['sequence_len']) / 1000 * sr / hl)
    min_dur = ((seq - 1) * hl + nfft) / sr      # one full 20 ms window
    info = sf.info(path)
    sr_in, n_in = info.samplerate, info.frames
    wins, owner = [], []
    for j, s in enumerate(sels):
        b, e = s['begin'], s['end']
        if e - b < min_dur:                     # too short for one window: centre-extend
            c = 0.5 * (b + e); b, e = c - min_dur / 2, c + min_dur / 2
        audio = resample_like_training(
            _read_mono(path, n_in, int(round(b * sr_in)), int(round(e * sr_in))), sr_in, sr)
        db = clip_to_db_spectrogram(audio, cfg)
        st, sc = window_scores(db, seq, WINDOW_STRIDE)
        for w0 in eval_window_starts(st, sc, WINDOW_MODE, TEST_TOPK, seq):
            wins.append(pad_window(minmax_normalize(np.array(db[w0:w0 + seq])), seq))
            owner.append(j)
    n_cls = len(cls['names'])
    if not wins:
        return np.zeros((0, n_cls))
    X = torch.from_numpy(np.stack(wins).astype(np.float32)).unsqueeze(1)
    P = np.concatenate([torch.softmax(model(X[b:b + batch].to(DEVICE)).float(), dim=1).cpu().numpy()
                        for b in range(0, len(X), batch)])
    out = np.zeros((len(sels), n_cls))
    np.add.at(out, np.array(owner), P)
    return out / np.bincount(owner, minlength=len(sels))[:, None]


def measure_frequencies(path, begin, end, floor_hz=None, snr_db=None, context_s=1.0, level_db=25):
    """(Low, High, Peak) frequency in Hz of a selection.
    Spectrogram at the native rate: Hann, ~375 Hz bins (1024 points at 384 kHz, as in the training
    Raven tables), 50 % overlap. Background = per-frequency median over the selection +- context_s
    (a wide context matters for the high-duty-cycle CF bats rhle/rhro, whose calls fill most frames
    of their own selection). Low/High = the contiguous band around the bin that stands highest above
    background, keeping bins >= snr_db above background AND within level_db of that bin's level;
    Peak = frequency of the loudest cell inside the band (Raven's Peak Freq rule). If no bin clears
    snr_db, Low = High = Peak. Checked on 400 annotated boxes (median error, bats): Peak 0.0 kHz
    (within one bin 65 %), Low 2.4 kHz, High 3.2 kHz. Weak spot: Low for rhle/rhro is often 40-55 kHz
    too low (faint CF calls; the band bridges into noise)."""
    floor_hz = INFER_FREQ_FLOOR_HZ if floor_hz is None else floor_hz
    snr_db = INFER_FREQ_SNR_DB if snr_db is None else snr_db
    info = sf.info(path)
    sr_in, n_in = info.samplerate, info.frames
    nfft = int(2 ** np.ceil(np.log2(sr_in / 375.0)))
    c, half = 0.5 * (begin + end), max(end - begin, 0.1) / 2
    a = max(0.0, c - half - context_s)
    x = _read_mono(path, n_in, int(round(a * sr_in)), int(round(min(n_in / sr_in, c + half + context_s) * sr_in)))
    if len(x) < 2 * nfft:
        x = np.pad(x, (0, 2 * nfft - len(x)))
    f, t, S = _ss.spectrogram(x, fs=sr_in, window='hann', nperseg=nfft, noverlap=nfft // 2, mode='psd')
    # Ceiling: next to Nyquist the recorder's anti-alias filter leaves an almost-zero background,
    # so any click there shows a huge "SNR" (a first version reported 192000 Hz for such clicks).
    keep = (f >= floor_hz) & (f <= min(INFER_FREQ_CEIL_HZ, 0.95 * sr_in / 2))
    if not keep.any():
        return float('nan'), float('nan'), float('nan')
    f, D = f[keep], 10 * np.log10(S[keep] + 1e-20)
    t = t + a
    inside = (t >= c - half) & (t <= c + half)
    Din = D[:, inside] if inside.any() else D
    snr = (Din - np.median(D, axis=1, keepdims=True)).max(axis=1)
    level = Din.max(axis=1)
    ip = int(np.argmax(snr))
    ok = (snr >= snr_db) & (level >= level[ip] - level_db)
    lo = hi = ip
    while lo > 0 and ok[lo - 1]:
        lo -= 1
    while hi < len(ok) - 1 and ok[hi + 1]:
        hi += 1
    peak = f[lo + int(np.argmax(level[lo:hi + 1]))]
    return float(f[lo]), float(f[hi]), float(peak)


# ---- clock time ------------------------------------------------------------------------------
_STAMP_RE = _re_inf.compile(r'(\d{8})[-_](\d{6})')
_CLIP_OFFSET_RE = _re_inf.compile(r'\d{8}[-_]\d{6}_(\d+)_(\d+)$')     # BatSpot clip: ..._<start ms>_<end ms>
_AUDIOMOTH_RE = _re_inf.compile(rb'Recorded at (\d{2}):(\d{2}):(\d{2}) (\d{2})/(\d{2})/(\d{4})')


def recording_start(path):
    """(datetime of the first sample, source). File name YYYYMMDD_HHMMSS (or -) first -- that is what
    Raven used for the training tables -- then the AudioMoth header comment, else (None, ...)."""
    stem = os.path.splitext(os.path.basename(path))[0]
    m = _STAMP_RE.search(stem)
    if m:
        try:
            t = _dt.datetime.strptime(m.group(1) + m.group(2), '%Y%m%d%H%M%S')
            off = _CLIP_OFFSET_RE.search(stem)
            if off:
                return t + _dt.timedelta(milliseconds=int(off.group(1))), 'file name + clip offset'
            return t, 'file name'
        except ValueError:
            pass
    try:
        with open(path, 'rb') as fh:
            m = _AUDIOMOTH_RE.search(fh.read(8192))
        if m:
            hh, mi, ss, dd, mo, yy = map(int, m.groups())
            return _dt.datetime(yy, mo, dd, hh, mi, ss), 'AudioMoth header'
    except (OSError, ValueError):
        pass
    return None, 'unknown -> clock times count from 00:00:00'


def clock_time(start, seconds):
    """Raven-style HH:MM:SS.ffff of `seconds` after `start` (wraps at midnight)."""
    base = 0.0 if start is None else start.hour * 3600 + start.minute * 60 + start.second + start.microsecond / 1e6
    t = int(round(((base + seconds) % 86400) * 1e4)) % (86400 * 10000)
    h, rem = divmod(t, 36000000)
    m, rem = divmod(rem, 600000)
    return f'{h:02d}:{m:02d}:{rem // 10000:02d}.{rem % 10000:04d}'


def process_recording(path, det, cls):
    """Detect, classify and measure every selection in one recording."""
    t_start, t_src = recording_start(path)
    t0, t1, p = scan_detector(path, det)
    sels = build_selections(t0, t1, p, INFER_DET_THRESHOLD, INFER_MERGE_GAP_S, INFER_MIN_WINDOWS)
    probs = classify_selections(path, sels, cls)
    for s, pr in zip(sels, probs):
        k = int(pr.argmax())
        s['species'], s['confidence'] = cls['names'][k], float(pr[k])
        s['is_noise'] = k == cls['noise_idx']
        s['low'], s['high'], s['peak'] = measure_frequencies(path, s['begin'], s['end'])
        s['begin_clock'], s['end_clock'] = clock_time(t_start, s['begin']), clock_time(t_start, s['end'])
    info = sf.info(path)
    return sels, {'duration': info.frames / info.samplerate, 'sr': info.samplerate,
                  'n_windows': len(p), 'p_max': float(p.max()) if len(p) else float('nan'),
                  'clock_source': t_src}


INFER_HEADER = ['Selection', 'name_of_file', 'Channel', 'Begin Time (s)', 'End Time (s)',
                'Begin Clock Time', 'End Clock Time', 'Low Freq (Hz)', 'High Freq (Hz)',
                'Peak Freq (Hz)', 'Delta Time (s)', 'Species detected', 'Confidence']


def _fmt_row(i, s):
    return [i, s['name'], 1, f'{s["begin"]:.4f}', f'{s["end"]:.4f}', s['begin_clock'], s['end_clock'],
            f'{s["low"]:.1f}', f'{s["high"]:.1f}', f'{s["peak"]:.1f}', f'{s["end"] - s["begin"]:.4f}',
            s['species'], f'{s["confidence"]:.4f}']


def write_detections(path, sels):
    """The requested comma-separated text file; Selection numbers run 1..N over the whole file."""
    with open(path, 'w', newline='') as fh:
        w = _csv.writer(fh)
        w.writerow(INFER_HEADER)
        for i, s in enumerate(sels, 1):
            w.writerow(_fmt_row(i, s))


def write_raven_table(path, sels):
    """Tab-separated Raven selection table for one recording (File -> Open Selection Table)."""
    cols = ['Selection', 'View', 'Channel', 'Begin Time (s)', 'End Time (s)', 'Low Freq (Hz)',
            'High Freq (Hz)', 'Begin Clock Time', 'End Clock Time', 'Peak Freq (Hz)', 'Delta Time (s)',
            'Species', 'Confidence']
    with open(path, 'w', newline='') as fh:
        fh.write('\t'.join(cols) + '\n')
        for i, s in enumerate(sels, 1):
            r = _fmt_row(i, s)
            fh.write('\t'.join(map(str, [i, 'Spectrogram 1', 1, r[3], r[4], r[7], r[8], r[5], r[6],
                                         r[9], r[10], r[11], r[12]])) + '\n')


print('Inference functions defined')

In [ ]:
# Cell 23: Run the detector -> classifier combo over every recording and write the detections file
import zipfile as _zipfile

_src = INFER_INPUT_DIR
if _src.lower().endswith('.zip'):
    _dst = os.path.join(WORKING_DIR, 'infer_input')
    with _zipfile.ZipFile(_src) as _z:
        _z.extractall(_dst)
    _src = _dst
assert os.path.isdir(_src), (f'INFER_INPUT_DIR not found: {INFER_INPUT_DIR!r} -- add the test data '
                             '(Add Data) and paste its path into Cell 21')
infer_wavs = sorted(p for p in glob.glob(os.path.join(_src, '**', '*'), recursive=True)
                    if p.lower().endswith('.wav') and '__MACOSX' not in p)
assert infer_wavs, f'no .wav files under {_src}'

_by_folder = defaultdict(list)
for _p in infer_wavs:
    _by_folder[os.path.dirname(os.path.relpath(_p, _src)) or '.'].append(_p)
print(f'{len(infer_wavs)} recordings in {len(_by_folder)} folder(s) under {_src}:')
for _f, _ps in sorted(_by_folder.items()):
    _infos = [sf.info(p) for p in _ps]
    print(f'  {_f:<30} {len(_ps):3d} files, {sum(i.duration for i in _infos)/60:6.1f} min, '
          f'sample rates {sorted({i.samplerate for i in _infos})}')
    _low = [os.path.basename(p) for p, i in zip(_ps, _infos) if i.samplerate < INFER_CLS['cfg']['sr']]
    if _low:
        print(f'    NOTE: {len(_low)} file(s) below {INFER_CLS["cfg"]["sr"]//1000} kHz -- calls above '
              f'their Nyquist frequency cannot be seen, e.g. {_low[:3]}')

infer_selections, infer_rejected, infer_failed, infer_files = [], [], [], []
_t = _time.time()
for _p in tqdm(infer_wavs, desc='recordings'):
    _rel = os.path.relpath(_p, _src)
    try:
        _sels, _info = process_recording(_p, INFER_DET, INFER_CLS)
    except Exception as _e:                      # one bad file must not stop the other 83
        infer_failed.append((_rel, repr(_e)))
        continue
    _folder = os.path.dirname(_rel) or '.'
    for _s in _sels:
        _s['name'] = _rel.replace(os.sep, '/') if INFER_NAME_WITH_FOLDER else os.path.basename(_p)
        _s['folder'], _s['path'] = _folder, _p
        (infer_rejected if (INFER_DROP_NOISE and _s['is_noise']) else infer_selections).append(_s)
    infer_files.append({'rel': _rel, 'folder': _folder, **_info,
                        'n_kept': sum(not (INFER_DROP_NOISE and s['is_noise']) for s in _sels),
                        'n_noise': sum(bool(s['is_noise']) for s in _sels)})
_elapsed = _time.time() - _t

write_detections(INFER_OUTPUT_FILE, infer_selections)
_rej_path = os.path.splitext(INFER_OUTPUT_FILE)[0] + '_rejected_noise.txt'
if INFER_DROP_NOISE:
    write_detections(_rej_path, infer_rejected)
if INFER_WRITE_RAVEN_TABLES:
    _rdir = os.path.join(WORKING_DIR, 'raven_tables')
    for _fi in infer_files:
        _out = os.path.join(_rdir, os.path.splitext(_fi['rel'])[0] + '.BatSpot.selections.txt')
        os.makedirs(os.path.dirname(_out), exist_ok=True)
        write_raven_table(_out, [s for s in infer_selections
                                 if os.path.relpath(s['path'], _src) == _fi['rel']])

# ---- report ---------------------------------------------------------------------------------
_audio_min = sum(f['duration'] for f in infer_files) / 60
print(f'\nProcessed {len(infer_files)}/{len(infer_wavs)} recordings ({_audio_min:.1f} min of audio) '
      f'in {_elapsed:.0f} s on {DEVICE}.')
print(f'Selections: {len(infer_selections)} kept'
      + (f', {len(infer_rejected)} labelled noise by the classifier (-> {os.path.basename(_rej_path)})'
         if INFER_DROP_NOISE else ''))
print(f'Detections file: {INFER_OUTPUT_FILE}')
if INFER_WRITE_RAVEN_TABLES:
    print(f'Raven tables:    {os.path.join(WORKING_DIR, "raven_tables")}/<folder>/<file>.BatSpot.selections.txt')
for _rel, _err in infer_failed:
    print(f'  FAILED {_rel}: {_err}')
_clock = Counter(f['clock_source'] for f in infer_files)
print('Clock-time source: ' + ', '.join(f'{k}: {v} file(s)' for k, v in _clock.items()))

_species = [n for n in INFER_CLS['names'] if n != 'noise']
print(f'\nSelections per folder and species detected:')
print(f'{"folder":<24}{"files":>6}{"w/ det":>7}' + ''.join(f'{s:>7}' for s in _species) + f'{"noise*":>8}')
for _f in sorted({fi['folder'] for fi in infer_files}):
    _fs = [fi for fi in infer_files if fi['folder'] == _f]
    _c = Counter(s['species'] for s in infer_selections if s['folder'] == _f)
    print(f'{_f[:23]:<24}{len(_fs):>6}{sum(fi["n_kept"] > 0 for fi in _fs):>7}'
          + ''.join(f'{_c.get(s, 0):>7}' for s in _species) + f'{sum(fi["n_noise"] for fi in _fs):>8}')
print('  (* noise = selections the classifier labelled noise; not in the main file when INFER_DROP_NOISE)')

# If the folders are named after species, show how often the folder's species was predicted.
_named = [f for f in sorted({fi['folder'] for fi in infer_files})
          if os.path.basename(f).strip().lower() in _species]
if _named:
    print('\nFolders named after a species -- share of kept selections labelled with that species:')
    for _f in _named:
        _sp = os.path.basename(_f).strip().lower()
        _sel = [s for s in infer_selections if s['folder'] == _f]
        if _sel:
            _hit = sum(s['species'] == _sp for s in _sel)
            print(f'  {_f:<24} {_hit:4d}/{len(_sel):<4d} = {_hit/len(_sel):.2f}   '
                  f'(median confidence {np.median([s["confidence"] for s in _sel]):.2f})')
        else:
            print(f'  {_f:<24} no kept selections')
    print('  This is a sanity check, not an accuracy: a folder can contain other species and noise.')

_none = [fi['rel'] for fi in infer_files if fi['n_kept'] == 0]
if _none:
    print(f'\n{len(_none)} recording(s) with no kept selection, e.g. {_none[:5]}')
    print('  If they contain calls, lower INFER_DET_THRESHOLD (Cell 21) or check them in Raven.')

print('\nFirst lines of the detections file:')
with open(INFER_OUTPUT_FILE) as _fh:
    for _i, _line in enumerate(_fh):
        if _i > 8:
            break
        print('  ' + _line.rstrip())

In [ ]:
# Cell 24 (optional): score the detections against your own Raven selection tables
#
# Set INFER_TRUTH_DIR (Cell 21) to a folder of Raven selection tables (.txt, tab-separated, with
# 'Begin Time (s)', 'End Time (s)' and a 'Species' column). A table belongs to a recording when its
# file name contains the recording's file stem -- e.g. '20260521_204000.Table.1.selections.txt' or
# 'acsh_devon_20260521_204000.txt' for '20260521_204000.WAV'; a table in the same sub-folder wins.
# Matching is by time overlap (any overlap), because a detected selection and a hand-drawn box rarely
# share exact edges. Precision is only meaningful if your tables mark EVERY call in the file.

def _read_truth_table(path):
    rows = []
    with open(path, newline='', encoding='utf-8', errors='replace') as fh:
        for r in _csv.DictReader(fh, delimiter='\t'):
            r = {(k or '').strip().lower(): (v or '').strip() for k, v in r.items()}
            if 'view' in r and 'spectrogram' not in r['view'].lower():
                continue                          # Raven repeats each box for the waveform view
            sp = (r.get('species') or r.get('species detected') or '').lower()
            try:
                rows.append((float(r['begin time (s)']), float(r['end time (s)']), sp))
            except (KeyError, ValueError):
                continue
    return rows


if INFER_TRUTH_DIR:
    _tables = sorted(glob.glob(os.path.join(INFER_TRUTH_DIR, '**', '*.txt'), recursive=True))
    print(f'{len(_tables)} truth table(s) under {INFER_TRUTH_DIR}')
    _tp_sp, _n_bat, _n_found, _n_noise, _n_noise_hit = [], 0, 0, 0, 0
    _n_det, _n_det_on_bat, _n_files = 0, 0, 0
    _per_folder = defaultdict(lambda: [0, 0, 0])      # bat boxes, found, species correct
    for _fi in infer_files:
        _stem = os.path.splitext(os.path.basename(_fi['rel']))[0].lower()
        _cand = [t for t in _tables if _stem in os.path.basename(t).lower()]
        _same = [t for t in _cand if os.path.relpath(os.path.dirname(t), INFER_TRUTH_DIR) == _fi['folder']]
        _cand = _same or _cand
        if not _cand:
            continue
        _n_files += 1
        _truth = [row for t in _cand for row in _read_truth_table(t)]
        _det = [s for s in infer_selections if os.path.relpath(s['path'], _src) == _fi['rel']]
        _ov = lambda s, b, e: min(s['end'], e) - max(s['begin'], b)
        for b, e, sp in _truth:
            _hits = [s for s in _det if _ov(s, b, e) > 0]
            if sp == 'noise':
                _n_noise += 1; _n_noise_hit += bool(_hits)
                continue
            _n_bat += 1; _per_folder[_fi['folder']][0] += 1
            if _hits:
                _n_found += 1; _per_folder[_fi['folder']][1] += 1
                _best = max(_hits, key=lambda s: _ov(s, b, e))
                _tp_sp.append((sp, _best['species']))
                _per_folder[_fi['folder']][2] += _best['species'] == sp
        _n_det += len(_det)
        _n_det_on_bat += sum(any(_ov(s, b, e) > 0 and sp != 'noise' for b, e, sp in _truth) for s in _det)

    if not _n_files:
        print('No table matched any recording -- table names must contain the recording file stem.')
    else:
        print(f'Scored {_n_files} recording(s) that have a truth table.')
        if _n_bat:
            print(f'  bat boxes found (any overlap with a kept selection): {_n_found}/{_n_bat} = {_n_found/_n_bat:.3f}')
        if _tp_sp:
            print(f'  species correct on found boxes: {sum(a == b for a, b in _tp_sp)}/{len(_tp_sp)} = '
                  f'{np.mean([a == b for a, b in _tp_sp]):.3f}')
        if _n_noise:
            print(f'  noise boxes hit by a kept selection (false alarms): {_n_noise_hit}/{_n_noise}')
        if _n_det:
            print(f'  kept selections overlapping a bat box: {_n_det_on_bat}/{_n_det} = {_n_det_on_bat/_n_det:.3f}'
                  '   (a precision only if the tables are exhaustive)')
        print(f'\n  {"folder":<24}{"bat boxes":>10}{"found":>8}{"species ok":>12}')
        for _f, (_a, _b, _c) in sorted(_per_folder.items()):
            print(f'  {_f[:23]:<24}{_a:>10}{_b:>8}{_c:>12}')
        if _tp_sp:
            _labs = sorted({a for a, _ in _tp_sp} | {b for _, b in _tp_sp})
            _cm = confusion_matrix([a for a, _ in _tp_sp], [b for _, b in _tp_sp], labels=_labs)
            print('\n  confusion on found boxes (rows = your label, columns = predicted):')
            print('  ' + ' ' * 8 + ''.join(f'{l[:6]:>7}' for l in _labs))
            for _l, _row in zip(_labs, _cm):
                print(f'  {_l[:7]:<8}' + ''.join(f'{v:>7}' for v in _row))
else:
    print('INFER_TRUTH_DIR not set (Cell 21) -- skipping the comparison with your own selection tables.')